# Politeness-LLM — Full Paper Pipeline (clean, 2026-10-12)

运行约定：**只用 cell 级 cmd+Enter，不要 Run all**。每个新运行时先跑 §1 Setup（Drive 授权需本人完成）。
数据在 Drive `$DATA_DIR`（raw/ append-only）；代码以 GitHub repo 为准（每节开头 `git pull`）。
大输出一律 `> /content/x.txt 2>&1; tail` 落盘，避免刷屏冻结页面。标记 [DONE] 的 cell 已完成，仅留档。
旧 notebook `2_full_paper_collect.ipynb` 保留为归档（含历史输出）。

## 1. Setup（每个新运行时必跑）

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
token = userdata.get('GH_TOKEN')
!git clone https://{token}@github.com/dukesky/politeness-llm.git 2>/dev/null || (cd politeness-llm && git pull)
%cd politeness-llm
!pip install -q -r requirements.txt
os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
os.environ['DATA_DIR'] = '/content/drive/MyDrive/Projects/research/politeness-llm'

## 2. 数据：pair 集冻结（一次性）

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]
# (c) build + FREEZE round-2 pairs (dl21/dl22: official scoreddocs top50 AND qrels; antique: qrels-only, grades 1-4 raw)
# NOTE: first run downloads msmarco-passage-v2 (~20GB) to the VM - slow, one-time per VM
!bash scripts/round2_build_pairs.sh

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]
# (d) frozen 40% flagship subsamples for round-2 datasets
!python scripts/make_flagship_sample.py --data-dir $DATA_DIR --datasets dl21,dl22,antique

### 2.1 v1（DL19/20）pair 构建（归档）

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]
!apt-get install -y -qq openjdk-21-jdk-headless > /dev/null 2>&1
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["JDK_HOME"] = os.environ["JAVA_HOME"]
!java -version   # 确认显示 21.x

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]
!pip install -q pyserini faiss-cpu

!python -m pyserini.search.lucene \
  --index msmarco-v1-passage --topics dl19-passage \
  --output $DATA_DIR/inputs/bm25_dl19.txt --bm25 --hits 100

!python -m pyserini.search.lucene \
  --index msmarco-v1-passage --topics dl20-passage \
  --output $DATA_DIR/inputs/bm25_dl20.txt --bm25 --hits 100

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]
# !python -m src.build_pairs --dataset dl19 \
#   --run-file $DATA_DIR/inputs/bm25_dl19.txt --top-k 50 \
#   --out $DATA_DIR/inputs/pairs_dl19.jsonl

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]
!python -m src.build_pairs --dataset dl20 \
  --run-file $DATA_DIR/inputs/bm25_dl20.txt --top-k 50 \
  --out $DATA_DIR/inputs/pairs_dl20.jsonl

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]
!wc -l $DATA_DIR/inputs/pairs_dl19.jsonl     # 行数 = pair 数
!head -2 $DATA_DIR/inputs/pairs_dl19.jsonl   # 肉眼看：query/passage 文本正常，无乱码截断

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]
!wc -l $DATA_DIR/inputs/pairs_dl20.jsonl     # 行数 = pair 数
!head -2 $DATA_DIR/inputs/pairs_dl20.jsonl   # 肉眼看：query/passage 文本正常，无乱码截断

## 3. 采集（全部完成；resume-by-key，重跑无副作用但勿无故触发 API）

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]
# (b) provider audit for all models incl. round-2 roster
!python scripts/check_providers.py

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]
# (e) dry-run: 5 pairs x 2 prompts x 1 model x 1 run
!python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl21.jsonl --models deepseek-v4.1-flash --dry-run

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]
# (e2) inspect dry-run records by hand before scaling (flat style: no indented blocks)
import json, glob, os
files = sorted(glob.glob(os.environ['DATA_DIR'] + "/raw/deepseek_deepseek-v4.1-flash/*.jsonl"))
recs = [json.loads(l) for fp in files for l in open(fp)]
print(len(files), 'files,', len(recs), 'records')
rows = [(r['prompt_id'], r['qid'], r['docid'], r['run'], r['score'], r['parse_ok'], r['finish_reason'], (r['raw_output'] or '')[:70], r.get('provider')) for r in recs]
print(*rows, sep='\n')

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]
# (f1a) TONIGHT (within $450): cheap tier WITHOUT gemini-3.8-flash (3.8 runs tomorrow after top-up)
!python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl21.jsonl --models deepseek-v4-flash,gemini-3.5-flash,deepseek-v4.1-flash
!python -m src.collect --pairs $DATA_DIR/inputs/pairs_antique.jsonl --models deepseek-v4-flash,gemini-3.5-flash,deepseek-v4.1-flash
!python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl22.jsonl --models deepseek-v4-flash,gemini-3.5-flash,deepseek-v4.1-flash

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]
# (f3) PLAN A+ TOMORROW: new flagships on OLD dl19/20 frozen 40% subsamples (generational pairs on shared data, ~$96)
!python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl19_flagship40.jsonl --models claude-opus-5,gpt-5.6-sol
!python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl20_flagship40.jsonl --models claude-opus-5,gpt-5.6-sol

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]
# (f1) CHEAP TIER: anchors + new cheap, full grid, 2 runs -- RUN ONLY AFTER BUDGET SIGN-OFF
!python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl21.jsonl --models deepseek-v4-flash,gemini-3.5-flash,deepseek-v4.1-flash,gemini-3.8-flash
!python -m src.collect --pairs $DATA_DIR/inputs/pairs_antique.jsonl --models deepseek-v4-flash,gemini-3.5-flash,deepseek-v4.1-flash,gemini-3.8-flash
!python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl22.jsonl --models deepseek-v4-flash,gemini-3.5-flash,deepseek-v4.1-flash,gemini-3.8-flash

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]
# (f2) FLAGSHIP TIER: 40% frozen subsample, 1 run -- RUN ONLY AFTER BUDGET SIGN-OFF
!python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl21_flagship40.jsonl --models claude-opus-5,gpt-5.6-sol
!python -m src.collect --pairs $DATA_DIR/inputs/pairs_antique_flagship40.jsonl --models claude-opus-5,gpt-5.6-sol
!python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl22_flagship40.jsonl --models claude-opus-5,gpt-5.6-sol

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]

# (h6g) FINAL collection of the project: 325 run-pool top-up pairs; L1+L3 for corrected judges, L3 for no-op control
!python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl21_runs_topup.jsonl --models v4-flash,v4.1-flash,gemini-3.5-flash --prompts L1,L3
!python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl21_runs_topup.jsonl --models gemini-3.8-flash --prompts L3

## 4. Parse 与完整性审计

In [ ]:
# (g1) rebuild parquet from ALL raw records (round 1 + round 2)
!python -m src.parse --data-dir $DATA_DIR

In [ ]:
# (g2) completeness audit: observed vs expected grid per model x dataset
import pandas as pd, os
df = pd.read_parquet(os.environ['DATA_DIR'] + '/derived/judgments.parquet')
g = df.groupby(['model_id','dataset'])
t = g.agg(n=('run','size'), variants=('prompt_id','nunique'), max_run=('run','max'), ok=('parse_ok','mean')).round(4)
t['pairs'] = g.apply(lambda x: x[['qid','docid']].drop_duplicates().shape[0])
exp = {'dl21': (1551, 622), 'dl22': (2146, 859), 'antique': (1985, 795), 'dl19': (1533, 614), 'dl20': (1965, 784)}
t['exp_cheap'] = [exp.get(ds,(None,None))[0]*30 if exp.get(ds) else None for _, ds in t.index]
t['exp_flag'] = [exp.get(ds,(None,None))[1]*15 if exp.get(ds) else None for _, ds in t.index]
print(t.to_string())

## 5. 验收协议：Step A 盲态预检 → 预测 commit → Step C 开箱；Batch 1R / Batch 4 / C4

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]
# (h1) Step A BLIND preflight: distributions only, NO kappa. dl22 stays sealed (distributions deferred to correction stage).
!python scripts/preflight_distributions.py --model deepseek/deepseek-v4-flash --datasets dl21
!python scripts/preflight_distributions.py --model deepseek/deepseek-v4-flash --datasets antique
!python scripts/preflight_distributions.py --model deepseek/deepseek-v4.1-flash --datasets dl21
!python scripts/preflight_distributions.py --model deepseek/deepseek-v4.1-flash --datasets antique
!python scripts/preflight_distributions.py --model google/gemini-3.5-flash --datasets dl21
!python scripts/preflight_distributions.py --model google/gemini-3.5-flash --datasets antique
!python scripts/preflight_distributions.py --model google/gemini-3.8-flash --datasets dl21
!python scripts/preflight_distributions.py --model google/gemini-3.8-flash --datasets antique
!python scripts/preflight_distributions.py --model anthropic/claude-opus-5 --datasets dl21
!python scripts/preflight_distributions.py --model anthropic/claude-opus-5 --datasets antique
!python scripts/preflight_distributions.py --model anthropic/claude-opus-5 --datasets dl19,dl20
!python scripts/preflight_distributions.py --model openai/gpt-5.6-sol --datasets dl21
!python scripts/preflight_distributions.py --model openai/gpt-5.6-sol --datasets antique
!python scripts/preflight_distributions.py --model openai/gpt-5.6-sol --datasets dl19,dl20

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]
# (h2) Step C kappa unboxing AFTER predictions commit 708f911. dl22 NEVER passed here (sealed).
!cd /content/politeness-llm/politeness-llm 2>/dev/null; git pull -q
!python scripts/validate_model.py --model deepseek/deepseek-v4-flash --datasets dl21
!python scripts/validate_model.py --model deepseek/deepseek-v4-flash --datasets antique
!python scripts/validate_model.py --model deepseek/deepseek-v4.1-flash --datasets dl21
!python scripts/validate_model.py --model deepseek/deepseek-v4.1-flash --datasets antique
!python scripts/validate_model.py --model google/gemini-3.5-flash --datasets dl21
!python scripts/validate_model.py --model google/gemini-3.5-flash --datasets antique
!python scripts/validate_model.py --model google/gemini-3.8-flash --datasets dl21
!python scripts/validate_model.py --model google/gemini-3.8-flash --datasets antique
!python scripts/validate_model.py --model anthropic/claude-opus-5 --datasets dl21
!python scripts/validate_model.py --model anthropic/claude-opus-5 --datasets antique
!python scripts/validate_model.py --model anthropic/claude-opus-5 --datasets dl19,dl20
!python scripts/validate_model.py --model openai/gpt-5.6-sol --datasets dl21
!python scripts/validate_model.py --model openai/gpt-5.6-sol --datasets antique
!python scripts/validate_model.py --model openai/gpt-5.6-sol --datasets dl19,dl20

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]
# (h3) Batch 1R statistical hardening (round-1 dl19/20 ONLY; authority assertion first)
!pip install -q statsmodels
!python scripts/batch1r_stats.py --data-dir $DATA_DIR --b 50 --n-perm 200
!python scripts/batch1r_stats.py --data-dir $DATA_DIR

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]
# (h4) Batch 4 operating-point correction: smoke then full. dl22 sealed (script has no unseal path).
!git pull -q
!python scripts/batch4_correction.py --data-dir $DATA_DIR --b 100
!python scripts/batch4_correction.py --data-dir $DATA_DIR

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]
# (h5a) C4 prep: show frozen rules + DL22 BLIND distribution preflight (distributions only, NO kappa)
!git pull -q
!cat $DATA_DIR/derived/correction_rules.json
!python scripts/preflight_distributions.py --model deepseek/deepseek-v4-flash --datasets dl22
!python scripts/preflight_distributions.py --model deepseek/deepseek-v4.1-flash --datasets dl22
!python scripts/preflight_distributions.py --model google/gemini-3.5-flash --datasets dl22
!python scripts/preflight_distributions.py --model google/gemini-3.8-flash --datasets dl22
!python scripts/preflight_distributions.py --model anthropic/claude-opus-5 --datasets dl22
!python scripts/preflight_distributions.py --model openai/gpt-5.6-sol --datasets dl22

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]
# (h5b) C4 UNSEALING — pre-registered at commit 80eacfc; the only dl22 kappa computation ever
!git pull -q
!python scripts/c4_unseal_dl22.py --data-dir $DATA_DIR --unseal-dl22

## 6. Batch 5：下游系统排名（gap-B）

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]
# (h6a) batch5 gap-B: pull review-fixed code, install pyserini, register the pool manifest
!git pull -q
!git log --oneline -1
!pip install -q pyserini==0.22.1 faiss-cpu
!python scripts/batch5_downstream.py fetch-runs --pyserini-pool

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]
# (h6b)) write the pool-generation snippet embedded in batch5_downstream.py to a file
import importlib, pathlib, sys
sys.path.insert(0, "/content/politeness-llm/scripts")
importlib.invalidate_caches()
import batch5_downstream as b5
n = pathlib.Path("/content/gen_pool.py").write_text(b5.PYSERINI_NOTEBOOK_SNIPPET)
print("wrote /content/gen_pool.py:", n, "bytes")

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]

# (h6c) generate the 15-config pyserini pool; runs to Drive; skip-if-exists makes it resumable
!python /content/gen_pool.py
!ls -la $DATA_DIR/runs/dl21/ | head -25

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]

# (h6f) coverage re-print, quiet: corpus+docstore cached now, log to file, show tail
!python scripts/batch5_downstream.py coverage --models deepseek/deepseek-v4-flash,deepseek/deepseek-v4.1-flash,google/gemini-3.5-flash,google/gemini-3.8-flash --runs 2 > /content/coverage.txt 2>&1; echo EXIT=$?; tail -90 /content/coverage.txt

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]

# (h6h) rebuild parquet with top-up records, then quiet coverage re-check
!python -m src.parse --data-dir $DATA_DIR
!python scripts/batch5_downstream.py coverage --models deepseek/deepseek-v4-flash,deepseek/deepseek-v4.1-flash,google/gemini-3.5-flash,google/gemini-3.8-flash --runs 2 > /content/coverage2.txt 2>&1; echo EXIT=$?; tail -45 /content/coverage2.txt

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]

# (h6l) evaluate with 90% CI + exact delta repr (commit 0e55074), archive to Drive
!git pull -q
!git log --oneline -1
!python scripts/batch5_downstream.py evaluate --models deepseek/deepseek-v4-flash,deepseek/deepseek-v4.1-flash,google/gemini-3.5-flash,google/gemini-3.8-flash > /content/eval2.txt 2>&1; echo EXIT=$?
!grep -A1 "delta_tau  " /content/eval2.txt
!cp /content/eval2.txt $DATA_DIR/derived/eval_batch5_stdout_ci90.txt

## 7. Application A：标注池扩充（batch6）

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]

# (h7a) batch6 A: pull code, validate on the real parquet (counts only), emit the blind registration block
!git pull -q
!git log --oneline -1
!nvidia-smi -L
!python scripts/batch6_poolext.py validate --datasets dl21,antique,dl22 > /content/b6_validate.txt 2>&1; echo VALIDATE_EXIT=$?; tail -70 /content/b6_validate.txt
!python scripts/batch6_poolext.py paste --datasets dl22,dl21,antique > /content/b6_paste.txt 2>&1; echo PASTE_EXIT=$?; wc -l /content/b6_paste.txt
!cp /content/b6_validate.txt /content/b6_paste.txt $DATA_DIR/derived/

In [ ]:
# [DONE 已完成，留档可重跑；结果见 Drive derived/ 与 paper/PREDICTIONS.md]

# (h7b) batch6 UNBOXING — registration cd5544d committed BEFORE this run; dl22 gated, then dl21 and antique
!git pull -q
!git log --oneline -1
!python scripts/batch6_poolext.py run --datasets dl22 --unseal-dl22 --sample-efficiency > /content/b6_run_dl22.txt 2>&1; echo DL22_EXIT=$?; tail -5 /content/b6_run_dl22.txt
!python scripts/batch6_poolext.py run --datasets dl21 --sample-efficiency > /content/b6_run_dl21.txt 2>&1; echo DL21_EXIT=$?; tail -3 /content/b6_run_dl21.txt
!python scripts/batch6_poolext.py run --datasets antique --sample-efficiency > /content/b6_run_antique.txt 2>&1; echo ANTIQUE_EXIT=$?; tail -3 /content/b6_run_antique.txt
!cp /content/b6_run_*.txt $DATA_DIR/derived/ && ls -la $DATA_DIR/derived/ | grep batch6

In [ ]:
# (h7c) batch6 follow-up [EXPLORATORY — not pre-registered; evaluator-requested]: 2x2 dial/map factorial, F1–ratio operating curve, F1-objective refit, nested bootstrap, E6' by queries
%%bash
cd /content/politeness-llm && git pull -q && git log --oneline -1
python scripts/batch6_poolext.py followup --data-dir $DATA_DIR --datasets dl22,dl21,antique --unseal-dl22 > $DATA_DIR/derived/b6_followup.txt 2>&1; echo FOLLOWUP_EXIT=$?
tail -4 $DATA_DIR/derived/b6_followup.txt


In [ ]:
# (h7d) batch6 followup2 [EXPLORATORY]: cross-fitted one-parameter L3 re-threshold baseline (RETHRESH_K / RETHRESH_F1), FROZEN dial/map curve verdicts, fold seeds 42-46, E6' fallback rate
%%bash
cd /content/politeness-llm && git pull -q && git log --oneline -1
python scripts/batch6_poolext.py followup2 --data-dir $DATA_DIR --datasets dl22,dl21,antique --unseal-dl22 --fold-seeds 42,43,44,45,46 > $DATA_DIR/derived/b6_followup2.txt 2>&1; echo FOLLOWUP2_EXIT=$?
tail -4 $DATA_DIR/derived/b6_followup2.txt


## 8. Application B：蒸馏训练（batch7，L4 GPU）

顺序：h8a 建标签 → h8b sweep → 写 B7 登记块并 commit → h8c 训练（可分段）→ h8d dl22 评测（门控）→ h8e 配对比较。

In [ ]:

# (h8a) batch7 B step 1: deps, CUDA smoke selftest, blind build-labels (train pool dl19/20/21; 3.8-flash dl21 only). No dl22.
!pip install -q -r requirements-batch7.txt 2>&1 | tail -2
!python scripts/batch7_distill.py selftest > /content/b7_selftest.txt 2>&1; echo SELFTEST_EXIT=$?; tail -4 /content/b7_selftest.txt
!python scripts/batch7_distill.py build-labels --judge deepseek/deepseek-v4-flash --train-datasets dl19,dl20,dl21 --rules $DATA_DIR/derived/correction_rules.json > /content/b7_labels_v4.txt 2>&1; echo V4_EXIT=$?; tail -25 /content/b7_labels_v4.txt
!python scripts/batch7_distill.py build-labels --judge google/gemini-3.5-flash --train-datasets dl19,dl20,dl21 --rules $DATA_DIR/derived/correction_rules.json > /content/b7_labels_g35.txt 2>&1; echo G35_EXIT=$?; tail -25 /content/b7_labels_g35.txt
!python scripts/batch7_distill.py build-labels --judge google/gemini-3.8-flash --train-datasets dl21 --rules $DATA_DIR/derived/correction_rules.json > /content/b7_labels_g38.txt 2>&1; echo G38_EXIT=$?; tail -25 /content/b7_labels_g38.txt
!cp /content/b7_*.txt $DATA_DIR/derived/

In [ ]:

# (h8b) batch7 B step 2: per-arm hyper-parameter sweep on the HUMAN set only (v4-flash pool), GPU; no dl22
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python scripts/batch7_distill.py sweep --arm minilm-raw --labels $DATA_DIR/derived/batch7/deepseek__deepseek-v4-flash/labels_HUMAN.jsonl --grid "epochs=3,5,8;lr=2e-5,3e-5,5e-5" --out $DATA_DIR/derived/batch7/sweep/minilm-raw/sweep.json > /content/b7_sweep_minilm.txt 2>&1; echo SWEEP1_EXIT=$?; tail -15 /content/b7_sweep_minilm.txt
!python scripts/batch7_distill.py sweep --arm msmarco-ce --labels $DATA_DIR/derived/batch7/deepseek__deepseek-v4-flash/labels_HUMAN.jsonl --grid "epochs=3,5,8;lr=2e-5,3e-5,5e-5" --out $DATA_DIR/derived/batch7/sweep/msmarco-ce/sweep.json > /content/b7_sweep_msmarco.txt 2>&1; echo SWEEP2_EXIT=$?; tail -15 /content/b7_sweep_msmarco.txt
!cp /content/b7_sweep_*.txt $DATA_DIR/derived/

In [ ]:
# (h8c) TRAIN — run ONLY after the B7 registration commit (sweep picks pasted into EP/LR). Resumable; one arm per session if needed.
%%bash
declare -A EP=( [minilm-raw]=5 [msmarco-ce]=3 )        # <- registered sweep picks (PREDICTIONS.md B7: minilm-raw 5/3e-5, msmarco-ce 3/3e-5)
declare -A LR=( [minilm-raw]=3e-5 [msmarco-ce]=3e-5 )  # <- registered sweep picks
SETS="HUMAN RAW CORR_FROZEN CORR_REFIT MIX_200 MIX_1000 CORR_REFIT_n200 CORR_REFIT_n1000"
for ARM in minilm-raw msmarco-ce; do
 for SLUG in deepseek__deepseek-v4-flash google__gemini-3.5-flash google__gemini-3.8-flash; do
  for SET in $SETS; do
   L="$DATA_DIR/derived/batch7/$SLUG/labels_$SET.jsonl"
   [ -f "$L" ] || { echo "NOTE: $L absent"; continue; }
   for SEED in 1 2 3; do
    OUT=/content/batch7_models/$ARM/$SLUG/${SET}_s$SEED
    if [ -f $OUT/train_log.json ]; then echo "exists: $OUT"; continue; fi
    python scripts/batch7_distill.py train --arm $ARM --labels "$L" --out $OUT --seed $SEED --epochs ${EP[$ARM]} --lr ${LR[$ARM]} --bs 32 --max-len 256 --objective regression > /content/b7_train_last.txt 2>&1 || { tail -30 /content/b7_train_last.txt; exit 1; }
    mkdir -p "$DATA_DIR/derived/batch7/train_logs/$ARM/$SLUG" && cp $OUT/train_log.json "$DATA_DIR/derived/batch7/train_logs/$ARM/$SLUG/${SET}_s$SEED.json"
    echo "done $OUT"
   done
  done
 done
done


In [ ]:
# (h8d) EVALUATE on dl22 — gated: requires heading '# B7 预注册 — 蒸馏训练' in a clean, committed PREDICTIONS.md
%%bash
for ARM in minilm-raw msmarco-ce; do
 for SLUG in deepseek__deepseek-v4-flash google__gemini-3.5-flash google__gemini-3.8-flash; do
  mkdir -p "$DATA_DIR/derived/batch7/eval/$ARM/$SLUG"
  for M in /content/batch7_models/$ARM/$SLUG/*_s[123]; do
   python scripts/batch7_distill.py evaluate --model-dir $M --dataset dl22 --unseal-dl22 --out "$DATA_DIR/derived/batch7/eval/$ARM/$SLUG/$(basename $M)_dl22.json" > /content/b7_eval_last.txt 2>&1 || { tail -20 /content/b7_eval_last.txt; exit 1; }
  done
 done
done
echo EVAL_DONE


In [ ]:
# (h8e) paired comparisons within one arm
%%bash
for ARM in minilm-raw msmarco-ce; do
 for SLUG in deepseek__deepseek-v4-flash google__gemini-3.5-flash google__gemini-3.8-flash; do
  E="$DATA_DIR/derived/batch7/eval/$ARM/$SLUG"
  s() { echo "$E/$1_s1_dl22.json,$E/$1_s2_dl22.json,$E/$1_s3_dl22.json"; }
  c() { [ -f "$E/$1_s1_dl22.json" ] && [ -f "$E/$2_s1_dl22.json" ] || { echo "NOTE: $1 or $2 absent"; return; }; python scripts/batch7_distill.py compare --a "$(s $1)" --b "$(s $2)" --out "$E/compare_$1_vs_$2.json" | tail -12; }
  for A in CORR_FROZEN CORR_REFIT MIX_200 MIX_1000 CORR_REFIT_n200 CORR_REFIT_n1000 HUMAN; do c $A RAW; done
  c CORR_REFIT_n200 MIX_200; c CORR_REFIT_n1000 MIX_1000; c CORR_REFIT HUMAN
 done
done


In [ ]:
# (h8f) B7 summary + registered P1–P8 adjudication (reporting only) -> Drive; CSVs feed the paper table builder
%%bash
cd /content/politeness-llm && git pull -q && git log --oneline -1
python scripts/batch7_summary.py --eval-root $DATA_DIR/derived/batch7/eval --csv-dir $DATA_DIR/derived/batch7 > $DATA_DIR/derived/batch7/b7_summary.txt 2>&1; echo EXIT=$?
grep -A60 'PRE-REGISTRATION ADJUDICATION' $DATA_DIR/derived/batch7/b7_summary.txt


In [ ]:
# (h8g) B7 post-hoc [EXPLORATORY — not pre-registered; evaluator-requested]: shift-calibrated RAW student, threshold-free endpoints, Kendall tau, teacher reference, label diagnostics (eval JSONs only, no GPU)
%%bash
cd /content/politeness-llm && git pull -q && git log --oneline -1
python scripts/batch7_posthoc.py run --eval-root $DATA_DIR/derived/batch7/eval --labels-root $DATA_DIR/derived/batch7 --b6-labels $DATA_DIR/derived/batch6_labels_dl22.parquet --out $DATA_DIR/derived/batch7/posthoc > /content/b7_posthoc_stdout.txt 2>&1; echo POSTHOC_EXIT=$?
sed -n '/COMPACT SUMMARY/,$p' $DATA_DIR/derived/batch7/posthoc/posthoc_report.txt


In [ ]:
# (h8h) B7 ABLATION [EXPLORATORY, registered as such in PREDICTIONS.md edbf67d]: build SHIFT_FULL / SHIFT_n1000 / DIAL_ONLY / MAP_ONLY on the registered universe (registered label files must stay byte-identical; the command checks)
%%bash
cd /content/politeness-llm && git pull -q && git log --oneline -1
for J in deepseek/deepseek-v4-flash google/gemini-3.5-flash; do
  python scripts/batch7_distill.py build-ablation --judge $J --data-dir $DATA_DIR > /content/b7_abl_$(echo $J | tr / _).txt 2>&1 || { tail -30 /content/b7_abl_$(echo $J | tr / _).txt; exit 1; }
  tail -25 /content/b7_abl_$(echo $J | tr / _).txt
done


In [ ]:
# (h8i) B7 ABLATION train: 2 judges x 4 sets x 3 seeds x 2 arms = 48 students (~20 min on L4); same registered hyper-parameters as h8c
%%bash
declare -A EP=( [minilm-raw]=5 [msmarco-ce]=3 )
declare -A LR=( [minilm-raw]=3e-5 [msmarco-ce]=3e-5 )
SETS="SHIFT_FULL SHIFT_n1000 DIAL_ONLY MAP_ONLY"
for ARM in minilm-raw msmarco-ce; do
 for SLUG in deepseek__deepseek-v4-flash google__gemini-3.5-flash; do
  for SET in $SETS; do
   L="$DATA_DIR/derived/batch7/$SLUG/labels_$SET.jsonl"
   [ -f "$L" ] || { echo "NOTE: $L absent"; continue; }
   for SEED in 1 2 3; do
    OUT=/content/batch7_models/$ARM/$SLUG/${SET}_s$SEED
    if [ -f $OUT/train_log.json ]; then echo "exists: $OUT"; continue; fi
    python scripts/batch7_distill.py train --arm $ARM --labels "$L" --out $OUT --seed $SEED --epochs ${EP[$ARM]} --lr ${LR[$ARM]} --bs 32 --max-len 256 --objective regression > /content/b7_train_last.txt 2>&1 || { tail -30 /content/b7_train_last.txt; exit 1; }
    mkdir -p "$DATA_DIR/derived/batch7/train_logs/$ARM/$SLUG" && cp $OUT/train_log.json "$DATA_DIR/derived/batch7/train_logs/$ARM/$SLUG/${SET}_s$SEED.json"
    echo "$(date +%H:%M) done $OUT"
   done
  done
 done
done
echo ABL_TRAIN_DONE


In [ ]:
# (h8j) B7 ABLATION evaluate on dl22 (same triple gate) + paired compares + summary
%%bash
cd /content/politeness-llm && git pull -q && git log --oneline -1 && git status --short | head -3
SETS="SHIFT_FULL SHIFT_n1000 DIAL_ONLY MAP_ONLY"
for ARM in minilm-raw msmarco-ce; do
 for SLUG in deepseek__deepseek-v4-flash google__gemini-3.5-flash; do
  E="$DATA_DIR/derived/batch7/eval/$ARM/$SLUG"; mkdir -p "$E"
  for SET in $SETS; do for SEED in 1 2 3; do
   M=/content/batch7_models/$ARM/$SLUG/${SET}_s$SEED; O="$E/${SET}_s${SEED}_dl22.json"
   [ -d $M ] || continue; [ -f "$O" ] && continue
   python scripts/batch7_distill.py evaluate --model-dir $M --dataset dl22 --unseal-dl22 --out "$O" > /content/b7_eval_last.txt 2>&1 || { tail -20 /content/b7_eval_last.txt; exit 1; }
  done; done
  s() { echo "$E/$1_s1_dl22.json,$E/$1_s2_dl22.json,$E/$1_s3_dl22.json"; }
  c() { [ -f "$E/$1_s1_dl22.json" ] && [ -f "$E/$2_s1_dl22.json" ] || { echo "NOTE: $1 or $2 absent"; return; }; python scripts/batch7_distill.py compare --a "$(s $1)" --b "$(s $2)" --out "$E/compare_$1_vs_$2.json" > "$E/compare_$1_vs_$2.txt" 2>&1 || tail -10 "$E/compare_$1_vs_$2.txt"; }
  for A in SHIFT_FULL SHIFT_n1000 DIAL_ONLY MAP_ONLY; do c $A RAW; c $A HUMAN; done
  c SHIFT_FULL CORR_FROZEN; c DIAL_ONLY CORR_FROZEN; c MAP_ONLY CORR_FROZEN; c DIAL_ONLY MAP_ONLY
  echo "$(date +%H:%M) evaluated+compared $ARM/$SLUG"
 done
done
python scripts/batch7_summary.py --eval-root $DATA_DIR/derived/batch7/eval --csv-dir $DATA_DIR/derived/batch7 > $DATA_DIR/derived/batch7/b7_summary_ablation.txt 2>&1; echo SUMMARY_EXIT=$?
sed -n '/ABLATION/,$p' $DATA_DIR/derived/batch7/b7_summary_ablation.txt


## 9. 归档：v1 分析与图（RecSys'26 Notes；原节 7–11、Paper Appendix、Camera Ready）

## 7. Analysis Distribution

In [ ]:
!python scripts/preflight_distributions.py --model openai/gpt-5.4-mini
!python scripts/preflight_distributions.py --model anthropic/claude-haiku-4.5

In [ ]:
!python scripts/preflight_distributions.py --model qwen/qwen3.7-plus
!python scripts/preflight_distributions.py --model google/gemini-3.5-flash

In [ ]:
!python scripts/preflight_distributions.py \
      --model openai/gpt-5.5 \
      --data-dir $DATA_DIR \
      --flagship-pairs $DATA_DIR/inputs/pairs_dl19_flagship40.jsonl \
                       $DATA_DIR/inputs/pairs_dl20_flagship40.jsonl

!python scripts/preflight_distributions.py \
      --model anthropic/claude-opus-4.8 \
      --data-dir $DATA_DIR \
      --flagship-pairs $DATA_DIR/inputs/pairs_dl19_flagship40.jsonl \
                       $DATA_DIR/inputs/pairs_dl20_flagship40.jsonl

!python scripts/preflight_distributions.py \
      --model google/gemini-3.1-pro-preview \
      --data-dir $DATA_DIR \
      --flagship-pairs $DATA_DIR/inputs/pairs_dl19_flagship40.jsonl \
                       $DATA_DIR/inputs/pairs_dl20_flagship40.jsonl


## 8. Compute Metrics

In [ ]:
!git pull

In [ ]:
!python scripts/validate_model.py --model deepseek/deepseek-v4-flash --data-dir $DATA_DIR
!python scripts/validate_model.py --model openai/gpt-5.4-mini --data-dir $DATA_DIR
!python scripts/validate_model.py --model anthropic/claude-haiku-4.5 --data-dir $DATA_DIR

In [ ]:
!python scripts/validate_model.py --model qwen/qwen3.7-plus --data-dir $DATA_DIR
!python scripts/validate_model.py --model google/gemini-3.5-flash --data-dir $DATA_DIR

In [ ]:
!python scripts/validate_model.py \
      --model openai/gpt-5.5 \
      --data-dir $DATA_DIR \
      --flagship-pairs $DATA_DIR/inputs/pairs_dl19_flagship40.jsonl \
                       $DATA_DIR/inputs/pairs_dl20_flagship40.jsonl

!python scripts/validate_model.py \
      --model anthropic/claude-opus-4.8 \
      --data-dir $DATA_DIR \
      --flagship-pairs $DATA_DIR/inputs/pairs_dl19_flagship40.jsonl \
                       $DATA_DIR/inputs/pairs_dl20_flagship40.jsonl

!python scripts/validate_model.py \
      --model google/gemini-3.1-pro-preview \
      --data-dir $DATA_DIR \
      --flagship-pairs $DATA_DIR/inputs/pairs_dl19_flagship40.jsonl \
                       $DATA_DIR/inputs/pairs_dl20_flagship40.jsonl

## 9. Spearman + Stable Analysis

In [ ]:
import pandas as pd
from scipy.stats import spearmanr

# 手动录入 8 模型的 Δ 和各档 D、Δκ(从 preflight 和 validation 结果)
# A(model,ℓ) = |Δ + D(ℓ)| − |Δ|;预测 A 与 Δκ 负相关
data = {
  # model: (Δ, {ℓ: (D, Δκ)})  ℓ ∈ L1,L2,L4,L5
  "deepseek":   (0.71, {"L1":(-0.073,0.0541),"L2":(-0.030,0.0278),"L4":(0.029,0.0341),"L5":(-0.018,0.0535)}),
  "mini":       (0.71, {"L1":(-0.0458,0.0064),"L2":(0.0049,-0.0147),"L4":(0.0000,-0.0028),"L5":(-0.0153,-0.0026)}),
  "haiku":      (0.52, {"L1":(-0.0108,-0.0006),"L2":(0.0132,-0.0099),"L4":(-0.0211,-0.0065),"L5":(-0.0102,-0.0100)}),
  "qwen":       (0.40, {"L1":(-0.0747,-0.0077),"L2":(0.0121,-0.0148),"L4":(0.0376,-0.0321),"L5":(-0.0084,-0.0039)}),
  "gemini-flash":(0.48,{"L1":(-0.0519,0.0134),"L2":(0.0005,-0.0072),"L4":(-0.0151,0.0118),"L5":(0.0382,-0.0474)}),
  "gpt-5.5":    (0.66, {"L1":(-0.0091,0.0011),"L2":(0.0169,-0.0037),"L4":(0.0305,-0.0060),"L5":(0.0300,-0.0081)}),
  "opus":       (0.35, {"L1":(-0.0381,0.0110),"L2":(0.0060,-0.0151),"L4":(-0.0017,-0.0087),"L5":(0.0253,-0.0119)}),
  # gemini-pro 是 reasoning 子组,单独算
}
rows=[]
for m,(D0,levels) in data.items():
    for l,(Dl,dk) in levels.items():
        A = abs(D0+Dl)-abs(D0)
        rows.append({"model":m,"level":l,"A":A,"dkappa":dk})
df=pd.DataFrame(rows)
rho,p=spearmanr(df["A"],df["dkappa"])
print(f"全样本(7非推理模型, n={len(df)}): Spearman ρ={rho:.3f}, p={p:.4f}")
print("机制预测 ρ<0(A越负=越贴近人类=Δκ越正)")

# 剔除 gemini-flash L5(L5_a 坏点污染)再算一次稳健性
df2=df[~((df["model"]=="gemini-flash")&(df["level"]=="L5"))]
rho2,p2=spearmanr(df2["A"],df2["dkappa"])
print(f"剔除 gemini-flash L5(L5_a异常): ρ={rho2:.3f}, p={p2:.4f}")

In [ ]:
import pandas as pd
from scipy.stats import spearmanr

# 28 个 (模型 × 档) 的 Δ、D(ℓ)、Δκ —— 全部取自 PREDICTIONS.md 各模型记录
# 格式 model: (Δ, {ℓ: (D(ℓ), Δκ(ℓ))})  ℓ ∈ L1,L2,L4,L5
data = {
  "deepseek-v4":     (0.71, {"L1":(-0.073, 0.0541), "L2":(-0.030, 0.0278), "L4":( 0.029, 0.0341), "L5":(-0.018, 0.0535)}),
  "gpt-mini":         (0.7085,{"L1":(-0.0458,0.0064), "L2":( 0.0049,-0.0147), "L4":( 0.0000,-0.0028), "L5":(-0.0153,-0.0026)}),
  "claude-haiku":        (0.5195,{"L1":(-0.0108,-0.0006),"L2":( 0.0132,-0.0099), "L4":(-0.0211,-0.0065), "L5":(-0.0102,-0.0100)}),
  "qwen3.7":         (0.3953,{"L1":(-0.0747,-0.0077),"L2":( 0.0121,-0.0148), "L4":( 0.0376,-0.0321), "L5":(-0.0084,-0.0039)}),
  "gemini-flash": (0.4798,{"L1":(-0.0519, 0.0134),"L2":( 0.0005,-0.0072), "L4":(-0.0151, 0.0118), "L5":( 0.0382,-0.0474)}),
  "gpt-5.5":      (0.6619,{"L1":(-0.0091, 0.0011),"L2":( 0.0169,-0.0037), "L4":( 0.0305,-0.0060), "L5":( 0.0300,-0.0081)}),
  "claude-opus":         (0.3522,{"L1":(-0.0381, 0.0110),"L2":( 0.0060,-0.0151), "L4":(-0.0017,-0.0087), "L5":( 0.0253,-0.0119)}),
}

rows = []
for m, (D0, levels) in data.items():
    for l, (Dl, dk) in levels.items():
        A = abs(D0 + Dl) - abs(D0)          # 对齐变化量:<0 表示漂移使工作点更靠近人类
        rows.append({"model": m, "level": l, "A": A, "dkappa": dk})
sp = pd.DataFrame(rows)   # 用 sp,避免和你原始 parquet 的 df 撞名

# ── 主检验:全样本 7 非推理模型 ──
rho, p = spearmanr(sp["A"], sp["dkappa"])
print(f"全样本 (7 非推理模型, n={len(sp)}): ρ={rho:.3f}, p={p:.4f}")

# ── 稳健性 a:剔除 gemini-flash L5 (L5_a 坏点) ──
a = sp[~((sp.model=="gemini-flash") & (sp.level=="L5"))]
print(f"稳健性a 剔 gemini-flash L5 (n={len(a)}): ρ={spearmanr(a.A,a.dkappa)[0]:.3f}, p={spearmanr(a.A,a.dkappa)[1]:.4f}")

# ── 稳健性 b:剔除 DeepSeek (机制假设的 non-blind 来源) ── 最关键
b = sp[sp.model!="deepseek"]
print(f"稳健性b 剔 DeepSeek (n={len(b)}): ρ={spearmanr(b.A,b.dkappa)[0]:.3f}, p={spearmanr(b.A,b.dkappa)[1]:.4f}")

# ── 稳健性 c:同时剔 DeepSeek + gemini-flash L5 ── 最保守
c = b[~((b.model=="gemini-flash") & (b.level=="L5"))]
print(f"稳健性c 剔 DeepSeek+geminiflashL5 (n={len(c)}): ρ={spearmanr(c.A,c.dkappa)[0]:.3f}, p={spearmanr(c.A,c.dkappa)[1]:.4f}")

# ── per-model 方向一致性:回应"同模型4档非独立"质疑 ──
print("\nper-model 方向符合机制的格数 (机制要求 A 与 Δκ 异号):")
for m in sp.model.unique():
    s = sp[sp.model==m]
    ok = ((s.A<0)&(s.dkappa>0)) | ((s.A>0)&(s.dkappa<0))
    print(f"  {m:14s}: {ok.sum()}/{len(s)}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import spearmanr

# ── 数据:每模型 (Δ, {档: (D(ℓ), Δκ(ℓ))}),与正文 4.2 / PREDICTIONS.md 一致 ──
data = {
  "DeepSeek V4 Flash": (0.71,   {"L1":(-0.073, 0.0541), "L2":(-0.030, 0.0278), "L4":( 0.029, 0.0341), "L5":(-0.018, 0.0535)}),
  "GPT-5.4-mini":      (0.7085, {"L1":(-0.0458,0.0064), "L2":( 0.0049,-0.0147),"L4":( 0.0000,-0.0028),"L5":(-0.0153,-0.0026)}),
  "Claude Haiku 4.5":  (0.5195, {"L1":(-0.0108,-0.0006),"L2":( 0.0132,-0.0099),"L4":(-0.0211,-0.0065),"L5":(-0.0102,-0.0100)}),
  "Qwen3.7-Plus":      (0.3953, {"L1":(-0.0747,-0.0077),"L2":( 0.0121,-0.0148),"L4":( 0.0376,-0.0321),"L5":(-0.0084,-0.0039)}),
  "Gemini 3.5 Flash":  (0.4798, {"L1":(-0.0519, 0.0134),"L2":( 0.0005,-0.0072),"L4":(-0.0151, 0.0118),"L5":( 0.0382,-0.0474)}),
  "GPT-5.5":           (0.6619, {"L1":(-0.0091, 0.0011),"L2":( 0.0169,-0.0037),"L4":( 0.0305,-0.0060),"L5":( 0.0300,-0.0081)}),
  "Claude Opus 4.8":   (0.3522, {"L1":(-0.0381, 0.0110),"L2":( 0.0060,-0.0151),"L4":(-0.0017,-0.0087),"L5":( 0.0253,-0.0119)}),
}

rows = []
for m, (D0, levels) in data.items():
    for l, (Dl, dk) in levels.items():
        A = abs(D0 + Dl) - abs(D0)
        rows.append({"model": m, "level": l, "A": A, "dk": dk})
sp = pd.DataFrame(rows)

rho, p = spearmanr(sp.A, sp.dk)

# ── 各模型标记(都蓝色,DeepSeek 红星突出)──
markers = {
    "GPT-5.4-mini":"o", "Claude Haiku 4.5":"s", "Qwen3.7-Plus":"^",
    "Gemini 3.5 Flash":"D", "GPT-5.5":"v", "Claude Opus 4.8":"P",
}

fig, ax = plt.subplots(figsize=(5.4, 4.2))

for m in data.keys():
    s = sp[sp.model==m]
    if m == "DeepSeek V4 Flash":
        ax.scatter(s.A, s.dk, s=130, c="crimson", marker="*",
                   edgecolors="black", linewidths=0.6,
                   label=m, zorder=3)
    else:
        ax.scatter(s.A, s.dk, s=50, c="steelblue", marker=markers[m],
                   alpha=0.8, edgecolors="none", label=m, zorder=2)

# 趋势线
z = np.polyfit(sp.A, sp.dk, 1)
xs = np.linspace(sp.A.min(), sp.A.max(), 100)
ax.plot(xs, z[0]*xs + z[1], "--", c="gray", lw=1.3, zorder=1, label="trend")

ax.axhline(0, c="lightgray", lw=0.7, zorder=0)
ax.axvline(0, c="lightgray", lw=0.7, zorder=0)

ax.set_xlabel(r"Alignment change $A(\ell)$", fontsize=11)
ax.set_ylabel(r"Agreement change $\kappa(\ell) - \kappa(L3)$", fontsize=11)
ax.set_title(rf"Spearman $\rho = {rho:.2f}$ ($p = {p:.4f}$)", fontsize=11)

# 图例移到图下方,横向 3 列,不挡数据
ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.16),
          ncol=3, frameon=False)
ax.tick_params(labelsize=9)
fig.tight_layout()

fig.savefig("/content/drive/MyDrive/Projects/research/politeness-llm/paper/fig_spearman.pdf",
            bbox_inches="tight")
print(f"已保存 fig_spearman.pdf;ρ={rho:.3f}, p={p:.4f}")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Δκ 数据:每模型在 L1-L5 相对 L3 的 κ 变化(L3=0)
# 来自你 Spearman 脚本的 dk 值(L1/L2/L4/L5)+ L3 基准=0
dk_data = {
  "DeepSeek V4 Flash":  {"L1":0.0541, "L2":0.0278, "L3":0.0, "L4":0.0341, "L5":0.0535},
  "GPT-5.4-mini":       {"L1":0.0064, "L2":-0.0147,"L3":0.0, "L4":-0.0028,"L5":-0.0026},
  "Claude Haiku 4.5":   {"L1":-0.0006,"L2":-0.0099,"L3":0.0, "L4":-0.0065,"L5":-0.0100},
  "Qwen3.7-Plus":       {"L1":-0.0077,"L2":-0.0148,"L3":0.0, "L4":-0.0321,"L5":-0.0039},
  "Gemini 3.5 Flash":   {"L1":0.0134, "L2":-0.0072,"L3":0.0, "L4":0.0118, "L5":-0.0474},
  "GPT-5.5":            {"L1":0.0011, "L2":-0.0037,"L3":0.0, "L4":-0.0060,"L5":-0.0081},
  "Claude Opus 4.8":    {"L1":0.0110, "L2":-0.0151,"L3":0.0, "L4":-0.0087,"L5":-0.0119},
}

levels = ["L1","L2","L3","L4","L5"]
x = range(len(levels))

fig, ax = plt.subplots(figsize=(5.4, 3.8))

for m, vals in dk_data.items():
    ys = [vals[l] for l in levels]
    if m == "DeepSeek V4 Flash":
        ax.plot(x, ys, "-o", c="crimson", lw=2.2, ms=7,
                label=m, zorder=3)
    else:
        ax.plot(x, ys, "-", c="steelblue", lw=1.0, alpha=0.55,
                marker="o", ms=4, label=m, zorder=2)

ax.axhline(0, c="gray", lw=0.8, ls="--", zorder=1)
ax.set_xticks(x)
ax.set_xticklabels(levels, fontsize=10)
ax.set_xlabel("Tone level (L1 rude $\\to$ L5 deferential)", fontsize=11)
ax.set_ylabel(r"$\Delta\kappa$ vs. neutral (L3)", fontsize=11)
ax.set_title("Tonal effect on agreement is model-dependent", fontsize=11)
ax.legend(fontsize=7.5, loc="upper center", bbox_to_anchor=(0.5, -0.16),
          ncol=3, frameon=False)
ax.tick_params(labelsize=9)
fig.tight_layout()
fig.savefig("/content/drive/MyDrive/Projects/research/politeness-llm/paper/fig_dkappa.pdf",
            bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

BASE = "/content/drive/MyDrive/Projects/research/politeness-llm"
df = pd.read_parquet(f"{BASE}/derived/judgments.parquet")

flagship = {"openai/gpt-5.5","anthropic/claude-opus-4.8","google/gemini-3.1-pro-preview"}

# 8 个模型:展示名 + 是否高亮
models = {
    "deepseek/deepseek-v4-flash":   "DeepSeek V4 Flash",
    "openai/gpt-5.4-mini":          "GPT-5.4-mini",
    "anthropic/claude-haiku-4.5":   "Claude Haiku 4.5",
    "qwen/qwen3.7-plus":            "Qwen3.7-Plus",
    "google/gemini-3.5-flash":      "Gemini 3.5 Flash",
    "openai/gpt-5.5":               "GPT-5.5",
    "anthropic/claude-opus-4.8":    "Claude Opus 4.8",
    "google/gemini-3.1-pro-preview":"Gemini 3.1 Pro",
}

levels = [1,2,3,4,5]
level_labels = ["L1","L2","L3","L4","L5"]
x = list(range(len(levels)))

# 给非 DeepSeek 模型分配淡色
cmap = plt.cm.tab10
colors = [cmap(i) for i in range(10)]

fig, ax = plt.subplots(figsize=(5.6, 4.2))

ci = 0
for mid, name in models.items():
    md = df[df.model_id==mid]
    means = md.groupby("politeness_level")["score"].mean()
    ys = [means.get(l, np.nan) for l in levels]
    if mid == "deepseek/deepseek-v4-flash":
        ax.plot(x, ys, "-o", c="crimson", lw=2.6, ms=7,
                label=name, zorder=5)
    else:
        ax.plot(x, ys, "-o", c=colors[ci], lw=1.2, ms=3.5, alpha=0.65,
                label=name, zorder=3)
        ci += 1

# 两条人类线
ax.axhline(0.9322, c="black", ls="--", lw=1.3, zorder=2)
ax.text(4.06, 0.9322, " human qrels\n (cost-eff.)", va="center", fontsize=8, c="black")
ax.axhline(0.9857, c="dimgray", ls=":", lw=1.2, zorder=2)
ax.text(4.06, 0.9857+0.03, " (flagship)", va="center", fontsize=8, c="dimgray")

ax.set_xticks(x)
ax.set_xticklabels(level_labels, fontsize=10)
ax.set_xlabel("Tone level (L1 rude $\\to$ L5 deferential)", fontsize=11)
ax.set_ylabel("Mean score (higher = more lenient)", fontsize=11)
ax.set_title("All judges are lenient; only DeepSeek shifts markedly with tone", fontsize=10)
ax.legend(fontsize=7.5, loc="upper center", bbox_to_anchor=(0.5, -0.15),
          ncol=4, frameon=False)
ax.tick_params(labelsize=9)
fig.tight_layout()
fig.savefig(f"{BASE}/paper/fig_opshift.pdf", bbox_inches="tight")
print("已保存 fig_opshift.pdf")

# 数值核对
for mid, name in models.items():
    md = df[df.model_id==mid]
    means = md.groupby("politeness_level")["score"].mean()
    print(f"{name:20}", {f"L{l}":round(means.get(l,float('nan')),3) for l in levels})
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

BASE = "/content/drive/MyDrive/Projects/research/politeness-llm"

levels = ["L1","L2","L3","L4","L5"]
x = list(range(len(levels)))

# DeepSeek 的 Δκ(相对 L3)——证明 κ 能大幅波动
dkappa_ds = [0.0541, 0.0278, 0.0, 0.0341, 0.0535]

# 所有 8 个模型的 NDCG@10(绝对值,来自 R1 表),转成相对 L3 的 ΔNDCG
ndcg_abs = {
    "DeepSeek V4 Flash":  [0.8105, 0.8258, 0.8212, 0.8227, 0.8215],
    "GPT-5.4-mini":       [0.7937, 0.8022, 0.7916, 0.7988, 0.7956],
    "Claude Haiku 4.5":   [0.8046, 0.8014, 0.7943, 0.8040, 0.8052],
    "Qwen3.7-Plus":       [0.7926, 0.8059, 0.7964, 0.7967, 0.7928],
    "Gemini 3.5 Flash":   [0.8029, 0.8074, 0.8099, 0.8096, 0.8065],
    "GPT-5.5":            [0.8088, 0.8056, 0.8064, 0.8164, 0.8114],
    "Claude Opus 4.8":    [0.8543, 0.8500, 0.8212, 0.8224, 0.8236],
    "Gemini 3.1 Pro":     [0.8153, 0.8177, 0.8224, 0.8152, 0.8175],
}

fig, ax = plt.subplots(figsize=(5.6, 4.0))

# 所有模型的 ΔNDCG(淡蓝,背景)
first = True
for name, vals in ndcg_abs.items():
    dndcg = [v - vals[2] for v in vals]
    ax.plot(x, dndcg, "-", c="steelblue", lw=1.0, alpha=0.45,
            marker="s", ms=3,
            label=r"$\Delta$NDCG@10 (all judges)" if first else None,
            zorder=2)
    first = False

# DeepSeek 的 Δκ(红,突出)
ax.plot(x, dkappa_ds, "-o", c="crimson", lw=1.5, ms=3,
        label=r"$\Delta\kappa$ (DeepSeek, most sensitive)", zorder=4)

ax.axhline(0, c="gray", ls="--", lw=0.9, zorder=1)

ax.set_xticks(x)
ax.set_xticklabels(levels, fontsize=10)
ax.set_xlabel("Tone level (L1 rude $\\to$ L5 deferential)", fontsize=11)
ax.set_ylabel("Change vs. neutral (L3)", fontsize=11)
ax.set_title("Tone perturbs calibration ($\\kappa$), not ranking (NDCG)", fontsize=10.5)
ax.legend(fontsize=8.5, loc="upper center", bbox_to_anchor=(0.5, -0.15),
          ncol=2, frameon=False)
ax.tick_params(labelsize=9)

# 注释:κ 波动大、NDCG 平
ax.annotate("structured\nU-shape", xy=(1, 0.0278), xytext=(1.25, 0.040),
            fontsize=8.5, c="crimson", ha="left",
            arrowprops=dict(arrowstyle="->", color="crimson", lw=1))
ax.annotate("all flat ($<0.033$)", xy=(3, 0.005), xytext=(3.1, 0.020),
            fontsize=8.5, c="steelblue", ha="left",
            arrowprops=dict(arrowstyle="->", color="steelblue", lw=1))

fig.tight_layout()
fig.savefig(f"{BASE}/paper/fig_kappa_ndcg.pdf", bbox_inches="tight")
print("已保存 fig_kappa_ndcg.pdf")
plt.show()

## 10. Side Case Analsysis

### 10.1 DeepSeek Abnormal Analsysis

In [ ]:
import pandas as pd, yaml
df = pd.read_parquet("/content/drive/MyDrive/Projects/research/politeness-llm/derived/judgments.parquet")

ds = df[(df.model_id=="deepseek/deepseek-v4-flash") & (df.politeness_level==2)]

print("DeepSeek L2 三改写分布对比:")
for pid in sorted(ds.prompt_id.unique()):
    s = ds[ds.prompt_id==pid]
    dist = s.score.value_counts(normalize=True).sort_index()
    print(f"\n{pid}: n={len(s)} mean={s.score.mean():.4f} "
          f"tok_out_med={s.tokens_out.median():.0f} parse_ok={s.parse_ok.mean():.3f}")
    print("  score: " + "  ".join(f"{k}={dist.get(k,0)*100:4.1f}%" for k in [0,1,2,3]))

# 看 L2_c 的 prompt 措辞,和 L2_a/b 对比
with open("config/prompts.yaml") as f:
    P = yaml.safe_load(f)
print("\n" + "="*60)
for item in P["variants"]:
    if item["prompt_id"] in ["L2_a","L2_b","L2_c"]:
        print(f"\n[{item['prompt_id']}] classifier={item['politeness_classifier_score']}")
        print("PREFIX:", repr(item["wrapper_prefix"]))
        print("SUFFIX:", repr(item["wrapper_suffix"]))

### 10.2 Gemini Reasoning Analysis

In [ ]:
gp = df[df.model_id=="google/gemini-3.1-pro-preview"]

print("gemini-3.1-pro 各语气档的 reasoning token 消耗:")
print(f"{'档':<6}{'mean_reasoning':>16}{'median':>10}{'mean_out':>10}")
for lvl in [1,2,3,4,5]:
    s = gp[gp.politeness_level==lvl]
    print(f"L{lvl:<5}{s.tokens_reasoning.mean():>16.1f}{s.tokens_reasoning.median():>10.0f}{s.tokens_out.mean():>10.1f}")

# 相对 L3 的变化
base = gp[gp.politeness_level==3].tokens_reasoning.mean()
print(f"\n相对 L3({base:.1f}) 的 reasoning token 变化:")
for lvl in [1,2,4,5]:
    m = gp[gp.politeness_level==lvl].tokens_reasoning.mean()
    print(f"  L{lvl}: {m-base:+.1f} ({(m-base)/base*100:+.1f}%)")

# 顺便相关:reasoning 多的时候 κ 是否不同?(探索)
from scipy.stats import spearmanr
per = gp.groupby("politeness_level").agg(rt=("tokens_reasoning","mean")).reset_index()
print("\n各档平均 reasoning token:", per.set_index("politeness_level").rt.round(1).to_dict())

In [ ]:
gp = df[df.model_id=="google/gemini-3.1-pro-preview"]

# 1. 看分布,不只看均值(均值可能被长尾拉高)
print("各档 reasoning token 分位数:")
print(f"{'档':<5}{'p25':>7}{'p50':>7}{'p75':>7}{'p95':>7}{'max':>7}")
for lvl in [1,2,3,4,5]:
    s = gp[gp.politeness_level==lvl].tokens_reasoning
    print(f"L{lvl:<4}{s.quantile(.25):>7.0f}{s.quantile(.5):>7.0f}"
          f"{s.quantile(.75):>7.0f}{s.quantile(.95):>7.0f}{s.max():>7.0f}")

# 2. 三改写一致性:L1 的 +120% 是真效应还是某条改写的 artifact(L5_a 的教训)
print("\nL1 三改写各自的 mean reasoning(查是否某条改写驱动):")
for pid in sorted(gp[gp.politeness_level==1].prompt_id.unique()):
    s = gp[gp.prompt_id==pid].tokens_reasoning
    print(f"  {pid}: mean={s.mean():.1f} median={s.median():.0f}")

## 11. Ranking Based Result + Analysis

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import kendalltau

df = pd.read_parquet("/content/drive/MyDrive/Projects/research/politeness-llm/derived/judgments.parquet")

# ── 需要 qrels 和 BM25 分数破平局 ──
# qrels: (dataset,qid,docid)->rel;  BM25: 用于分数相同时破平局
# 先确认 df 里有没有 bm25 列；多半没有，需要从 inputs/pairs 读
print("df 列:", [c for c in df.columns])
# 若无 bm25，用 docid 的稳定顺序破平局（次优但确定性）；理想是用 BM25 分。

# 读 qrels（你的 metrics 脚本能下载，这里假设 pairs 文件里带 qrel 标签）
# 先探明 pairs 文件结构
import json
DATA="/content/drive/MyDrive/Projects/research/politeness-llm/inputs"
sample = [json.loads(l) for i,l in enumerate(open(f"{DATA}/pairs_dl19.jsonl")) if i<2]
print("pairs 样本:", sample)

In [ ]:
import os, glob
DATA = "/content/drive/MyDrive/Projects/research/politeness-llm/inputs"
print("inputs/ 下的文件:")
for f in sorted(glob.glob(f"{DATA}/*")):
    sz = os.path.getsize(f)
    print(f"  {os.path.basename(f):40s} {sz/1024:.1f} KB")

# 看看有没有 BM25 run 文件,瞄一眼格式
for f in glob.glob(f"{DATA}/*"):
    name = os.path.basename(f).lower()
    if "bm25" in name or "run" in name or name.endswith(".trec") or name.endswith(".tsv"):
        print(f"\n疑似 BM25 run: {f}")
        with open(f) as fh:
            for i, line in enumerate(fh):
                print("  ", line.rstrip())
                if i >= 2: break

In [ ]:
import pandas as pd, numpy as np, urllib.request, os
from scipy.stats import kendalltau

BASE = "/content/drive/MyDrive/Projects/research/politeness-llm"
df = pd.read_parquet(f"{BASE}/derived/judgments.parquet")

# ── 1. qrels(带 User-Agent) ──
def load_qrels(ds):
    url = f"https://trec.nist.gov/data/deep/{'2019' if ds=='dl19' else '2020'}qrels-pass.txt"
    local = f"/tmp/{ds}_qrels.txt"
    if not os.path.exists(local):
        req = urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
        with urllib.request.urlopen(req) as r, open(local,"wb") as f:
            f.write(r.read())
    q = {}
    for line in open(local):
        qid,_,docid,rel = line.split()
        q[(ds,qid,docid)] = int(rel)
    return q
qrels = {**load_qrels("dl19"), **load_qrels("dl20")}

# ── 2. BM25(破平局) ──
bm25 = {}
for ds in ["dl19","dl20"]:
    for line in open(f"{BASE}/inputs/bm25_{ds}.txt"):
        qid,_,docid,rank,score,_ = line.split()
        bm25[(ds,qid,docid)] = float(score)

# ── 3. NDCG@10 ──
def dcg(rels): return sum((2**r-1)/np.log2(i+2) for i,r in enumerate(rels))
def ndcg10(ranked, ds, qid):
    rels = [qrels.get((ds,qid,d),0) for d in ranked[:10]]
    ideal = sorted([qrels.get((ds,qid,d),0) for d in ranked], reverse=True)[:10]
    idcg = dcg(ideal)
    return dcg(rels)/idcg if idcg>0 else 0.0

# ── 4. 派生排序:双 run 取均分,score 降序,BM25 破平局 ──
def get_ranking(sub, ds, qid):
    g = sub.groupby("docid")["score"].mean().reset_index()
    g["bm25"] = g["docid"].map(lambda d: bm25.get((ds,qid,d),0.0))
    g = g.sort_values(["score","bm25"], ascending=[False,False])
    return g["docid"].tolist()

# ── R1 ──
print("="*72); print("R1: 语气对派生排序质量 NDCG@10"); print("="*72)
r1_rows=[]
for model in sorted(df.model_id.unique()):
    md=df[df.model_id==model]; lv={}
    for lvl in [1,2,3,4,5]:
        ld=md[md.politeness_level==lvl]; ndcgs=[]
        for (ds,qid),sub in ld.groupby(["dataset","qid"]):
            ndcgs.append(ndcg10(get_ranking(sub,ds,qid),ds,qid))
        lv[lvl]=np.mean(ndcgs)
    b=lv[3]
    r1_rows.append({"model":model.split("/")[-1],**{f"L{l}":lv[l] for l in [1,2,3,4,5]},
                    "dN_L1":lv[1]-b,"dN_L5":lv[5]-b})
r1=pd.DataFrame(r1_rows)
pd.set_option("display.float_format",lambda x:f"{x:.4f}")
print(r1.to_string(index=False))

# ── R2 ──
print("\n"+"="*72)
print("R2: 各档 vs L3 的 Kendall τ(τ高=排序顺序不变=语气动松紧不动眼光)")
print("="*72)
r2_rows=[]
for model in sorted(df.model_id.unique()):
    md=df[df.model_id==model]; taus={1:[],2:[],4:[],5:[]}
    for (ds,qid),_ in md.groupby(["dataset","qid"]):
        l3=md[(md.politeness_level==3)&(md.dataset==ds)&(md.qid==qid)]
        if len(l3)==0: continue
        rank3={d:i for i,d in enumerate(get_ranking(l3,ds,qid))}
        for lvl in [1,2,4,5]:
            ll=md[(md.politeness_level==lvl)&(md.dataset==ds)&(md.qid==qid)]
            if len(ll)==0: continue
            common=[d for d in get_ranking(ll,ds,qid) if d in rank3]
            if len(common)<3: continue
            tau,_=kendalltau(list(range(len(common))),[rank3[d] for d in common])
            if not np.isnan(tau): taus[lvl].append(tau)
    r2_rows.append({"model":model.split("/")[-1],**{f"tau_L{l}":np.mean(taus[l]) for l in [1,2,4,5]}})
print(pd.DataFrame(r2_rows).to_string(index=False))

## Paper Appendix

In [ ]:
# === 生成 15 变体的 polite-guard 校准分数(用于附录表) ===
import os, glob, json
import torch
from transformers import pipeline

BASE = "."

# 1. 找 prompts 配置文件(yaml/json),先定位
for f in glob.glob(f"{BASE}/config/*", recursive=True):
    name = os.path.basename(f).lower()
    if "prompt" in name and (name.endswith(".yaml") or name.endswith(".yml") or name.endswith(".json")):
        print("找到 prompt 配置:", f)

In [ ]:
import yaml, torch
from transformers import pipeline

BASE = "."

# 1. 加载 polite-guard(T4 用 fp16)
clf = pipeline("text-classification", model="Intel/polite-guard",
               top_k=None, device=0,
               torch_dtype=torch.float16)

# 2. 看原始输出结构:label 名 + 概率
sample = "Score this passage. Don't waste my time with explanations."
out = clf(sample)
print("原始输出:")
print(out)

# 3. 读 prompts.yaml,看结构(确认 wrapper 字段名)
with open(f"{BASE}/config/prompts.yaml") as f:
    cfg = yaml.safe_load(f)
print("\nprompts.yaml 顶层 keys:", list(cfg.keys()) if isinstance(cfg, dict) else type(cfg))
# 打印第一条看结构
print(json.dumps(cfg, ensure_ascii=False, indent=2)[:1500])

In [ ]:
import yaml, json
BASE = "."
with open(f"{BASE}/config/prompts.yaml") as f:
    cfg = yaml.safe_load(f)

rows = []
for v in cfg["variants"]:
    rows.append({
        "id": v["prompt_id"],
        "level": v["politeness_level"],
        "score": v["politeness_classifier_score"],
        "prefix": v["wrapper_prefix"].strip(),
    })

# 打印成表 + 每档均值核对
print(f"{'id':6} {'lvl':3} {'score':6}  prefix")
print("-"*80)
from collections import defaultdict
by_level = defaultdict(list)
for r in rows:
    by_level[r["level"]].append(r["score"])
    print(f"{r['id']:6} {r['level']:<3} {r['score']:<6.3f}  {r['prefix'][:60]}")

print("\n每档均值(核对是否 = 0.00/0.68/1.00/2.33/3.00):")
for lvl in sorted(by_level):
    vals = by_level[lvl]
    print(f"  L{lvl}: mean={sum(vals)/len(vals):.3f}  values={[round(x,3) for x in vals]}")

# 顺便存成 JSON,方便我做表
print("\n=== JSON(贴回给我)===")
print(json.dumps(rows, ensure_ascii=False))

In [ ]:
import pandas as pd
BASE = "/content/drive/MyDrive/Projects/research/politeness-llm"
df = pd.read_parquet(f"{BASE}/derived/judgments.parquet")

# qrels 均分:便宜档全量 0.9322,旗舰 40% 子样本 0.9857
QREL_MEAN = {"cheap": 0.9322, "flagship": 0.9857}
flagship = {"openai/gpt-5.5","anthropic/claude-opus-4.8","google/gemini-3.1-pro-preview"}

print(f"{'model':32} {'Δ':>8} {'D(L1)':>8} {'D(L2)':>8} {'D(L4)':>8} {'D(L5)':>8}")
for m in sorted(df.model_id.unique()):
    md = df[df.model_id==m]
    mean_by_lvl = md.groupby("politeness_level")["score"].mean()
    if 3 not in mean_by_lvl: continue
    s_L3 = mean_by_lvl[3]
    qrel = QREL_MEAN["flagship"] if m in flagship else QREL_MEAN["cheap"]
    delta = s_L3 - qrel
    D = {l: mean_by_lvl.get(l, float('nan')) - s_L3 for l in [1,2,4,5]}
    print(f"{m:32} {delta:+8.3f} {D[1]:+8.3f} {D[2]:+8.3f} {D[4]:+8.3f} {D[5]:+8.3f}")

## Camera Ready Update

In [ ]:
# ============================================================
# AUDIT 1: Recompute A and Δκ from the final parquet
#          + exact model-block permutation
# ============================================================

import os
import itertools
import numpy as np
import pandas as pd
import ir_datasets

from scipy.stats import spearmanr, binomtest
from sklearn.metrics import cohen_kappa_score

BASE = "/content/drive/MyDrive/Projects/research/politeness-llm"
PARQUET = f"{BASE}/derived/judgments.parquet"

# ---------- 1. Load final data ----------
raw = pd.read_parquet(PARQUET).copy()

for col in ["dataset", "qid", "docid", "model_id", "prompt_id"]:
    raw[col] = raw[col].astype(str)

print("=" * 72)
print("FINAL PARQUET AUDIT")
print("=" * 72)
print("shape:", raw.shape)
print("models:", sorted(raw.model_id.unique()))

key = ["model_id", "prompt_id", "dataset", "qid", "docid", "run"]
n_duplicates = raw.duplicated(key).sum()
print("duplicate model/prompt/pair/run rows:", n_duplicates)

if n_duplicates:
    raise ValueError(
        "Duplicate collection rows detected. Do not continue until inspected."
    )

# ---------- 2. Restrict all flagship models to the same frozen pairs ----------
reasoning_model = "google/gemini-3.1-pro-preview"

flagship_models = {
    "openai/gpt-5.5",
    "anthropic/claude-opus-4.8",
    reasoning_model,
}

# Gemini Pro was collected only on the intended frozen 40% sample,
# so use its pair set to filter the other flagship models.
frozen_pairs = (
    raw.loc[
        raw.model_id == reasoning_model,
        ["dataset", "qid", "docid"]
    ]
    .drop_duplicates()
    .assign(_frozen_pair=True)
)

raw = raw.merge(
    frozen_pairs,
    on=["dataset", "qid", "docid"],
    how="left",
)

raw = raw[
    (~raw.model_id.isin(flagship_models))
    | raw["_frozen_pair"].eq(True)
].drop(columns="_frozen_pair")

# ---------- 3. Load qrels ----------
qrels = {}

datasets = {
    "dl19": "msmarco-passage/trec-dl-2019/judged",
    "dl20": "msmarco-passage/trec-dl-2020/judged",
}

for dataset_name, ir_name in datasets.items():
    for qrel in ir_datasets.load(ir_name).qrels_iter():
        qrels[
            (
                dataset_name,
                str(qrel.query_id),
                str(qrel.doc_id),
            )
        ] = int(qrel.relevance)

raw["human"] = [
    qrels.get((dataset, qid, docid))
    for dataset, qid, docid in zip(
        raw.dataset,
        raw.qid,
        raw.docid,
    )
]

ok = raw[
    raw.parse_ok
    & raw.score.notna()
    & raw.human.notna()
].copy()

print("\nRows after flagship filtering and validity checks:", len(ok))
print("Rows missing human labels:", raw.human.isna().sum())

print("\nPer-model data coverage:")
coverage = (
    ok.groupby("model_id")
    .agg(
        rows=("score", "size"),
        prompts=("prompt_id", "nunique"),
        pairs=("docid", "count"),
        qids=("qid", "nunique"),
        runs=("run", "nunique"),
    )
)
print(coverage.to_string())

# ---------- 4. Compute κ exactly as in validation:
#              κ per prompt, then average the 3 paraphrases ----------
per_prompt_rows = []

for (model, level, prompt_id), group in ok.groupby(
    ["model_id", "politeness_level", "prompt_id"]
):
    kappa = cohen_kappa_score(
        group.human.astype(int),
        group.score.astype(int),
        weights="linear",
    )

    per_prompt_rows.append(
        {
            "model": model,
            "level": int(level),
            "prompt_id": prompt_id,
            "kappa": kappa,
        }
    )

per_prompt = pd.DataFrame(per_prompt_rows)

level_kappa = (
    per_prompt.groupby(["model", "level"])
    .kappa.mean()
)

# Mean score used for Δ and D
level_mean_score = (
    ok.groupby(["model_id", "politeness_level"])
    .score.mean()
)

# Human mean from unique pairs for each model
unique_pairs = ok[
    ["model_id", "dataset", "qid", "docid", "human"]
].drop_duplicates()

human_mean = (
    unique_pairs.groupby("model_id")
    .human.mean()
)

# ---------- 5. Construct the 28 final cells ----------
nonreasoning_models = [
    "deepseek/deepseek-v4-flash",
    "openai/gpt-5.4-mini",
    "anthropic/claude-haiku-4.5",
    "qwen/qwen3.7-plus",
    "google/gemini-3.5-flash",
    "openai/gpt-5.5",
    "anthropic/claude-opus-4.8",
]

contrast_levels = [1, 2, 4, 5]
rows = []

for model in nonreasoning_models:
    neutral_score = level_mean_score.loc[(model, 3)]
    neutral_kappa = level_kappa.loc[(model, 3)]

    delta = neutral_score - human_mean.loc[model]

    for level in contrast_levels:
        drift = (
            level_mean_score.loc[(model, level)]
            - neutral_score
        )

        alignment_change = (
            abs(delta + drift)
            - abs(delta)
        )

        dkappa = (
            level_kappa.loc[(model, level)]
            - neutral_kappa
        )

        rows.append(
            {
                "model": model,
                "level": f"L{level}",
                "Delta": delta,
                "D": drift,
                "A": alignment_change,
                "dkappa": dkappa,
            }
        )

sp = pd.DataFrame(rows)

assert len(sp) == 28, f"Expected 28 cells, found {len(sp)}"

print("\n" + "=" * 72)
print("RECOMPUTED 28 CELLS")
print("=" * 72)
print(
    sp.to_string(
        index=False,
        formatters={
            "Delta": "{:+.6f}".format,
            "D": "{:+.6f}".format,
            "A": "{:+.6f}".format,
            "dkappa": "{:+.6f}".format,
        },
    )
)

# ---------- 6. Pooled descriptive Spearman ----------
rho, nominal_p = spearmanr(sp.A, sp.dkappa)

print("\n" + "=" * 72)
print("CLUSTER-AWARE RESULTS")
print("=" * 72)
print(
    f"Pooled descriptive Spearman: "
    f"rho={rho:.6f}, nominal p={nominal_p:.8f}"
)

# ---------- 7. Correct leave-one-model-out ----------
print("\nLeave-one-model-out:")
for omitted_model in nonreasoning_models:
    subset = sp[sp.model != omitted_model]
    result = spearmanr(subset.A, subset.dkappa)

    print(
        f"omit {omitted_model:38s} "
        f"n={len(subset):2d} "
        f"rho={result.statistic:+.6f}"
    )

# ---------- 8. Within-model associations ----------
model_rhos = {}

print("\nWithin-model Spearman correlations:")
for model in nonreasoning_models:
    subset = sp[sp.model == model]
    model_rho = spearmanr(
        subset.A,
        subset.dkappa,
    ).statistic

    model_rhos[model] = model_rho

    print(
        f"{model:42s} "
        f"rho={model_rho:+.6f}"
    )

n_negative = sum(value < 0 for value in model_rhos.values())

sign_result = binomtest(
    n_negative,
    len(model_rhos),
    p=0.5,
    alternative="greater",  # pre-specified negative direction
)

print(
    f"\nNegative within-model associations: "
    f"{n_negative}/{len(model_rhos)}"
)
print(
    f"Exact one-sided model-level sign-test p="
    f"{sign_result.pvalue:.8f}"
)

# ---------- 9. Exact model-block permutation ----------
# Each model's four A cells remain together.
# Entire four-cell A blocks are permuted across model outcome blocks.

levels = ["L1", "L2", "L4", "L5"]

A_blocks = {
    model: (
        sp[sp.model == model]
        .set_index("level")
        .loc[levels, "A"]
        .to_numpy()
    )
    for model in nonreasoning_models
}

Y_blocks = {
    model: (
        sp[sp.model == model]
        .set_index("level")
        .loc[levels, "dkappa"]
        .to_numpy()
    )
    for model in nonreasoning_models
}

observed_rho = spearmanr(
    np.concatenate(
        [A_blocks[m] for m in nonreasoning_models]
    ),
    np.concatenate(
        [Y_blocks[m] for m in nonreasoning_models]
    ),
).statistic

permuted_rhos = []

for permutation in itertools.permutations(nonreasoning_models):
    permuted_A = np.concatenate(
        [A_blocks[m] for m in permutation]
    )

    fixed_Y = np.concatenate(
        [Y_blocks[m] for m in nonreasoning_models]
    )

    permuted_rhos.append(
        spearmanr(permuted_A, fixed_Y).statistic
    )

permuted_rhos = np.asarray(permuted_rhos)

one_sided_p = np.mean(
    permuted_rhos <= observed_rho + 1e-15
)

two_sided_p = np.mean(
    np.abs(permuted_rhos)
    >= abs(observed_rho) - 1e-15
)

print(
    f"\nExact model-block permutations: "
    f"{len(permuted_rhos)}"
)
print(
    f"Observed rho: {observed_rho:+.6f}"
)
print(
    f"Exact one-sided block-permutation p: "
    f"{one_sided_p:.8f}"
)
print(
    f"Exact two-sided block-permutation p: "
    f"{two_sided_p:.8f}"
)

In [ ]:
# ============================================================
# AUDIT 2:
# 1. Correct robustness slices using the final 28 cells
# 2. Query-disjoint cross-fitting:
#    estimate A on one query fold and Δκ on the other fold
# 3. Exact model-block permutation on the cross-fitted results
#
# This cell assumes AUDIT 1 has already been run and that
# `ok`, `sp`, and `nonreasoning_models` are in memory.
# ============================================================

import hashlib
import itertools
import numpy as np
import pandas as pd

from scipy.stats import spearmanr, binomtest
from sklearn.metrics import cohen_kappa_score


# ------------------------------------------------------------
# Part A. Correct robustness slices from the final 28 cells
# ------------------------------------------------------------

def report_spearman(name, frame):
    result = spearmanr(frame.A, frame.dkappa)
    print(
        f"{name:42s} "
        f"n={len(frame):2d} "
        f"rho={result.statistic:+.6f} "
        f"nominal_p={result.pvalue:.8f}"
    )


print("=" * 76)
print("CORRECTED ROBUSTNESS SLICES")
print("=" * 76)

report_spearman(
    "All non-reasoning models",
    sp,
)

without_anomalous = sp[
    ~(
        (sp.model == "google/gemini-3.5-flash")
        & (sp.level == "L5")
    )
]

report_spearman(
    "Excluding anomalous Gemini L5",
    without_anomalous,
)

without_deepseek = sp[
    sp.model != "deepseek/deepseek-v4-flash"
]

report_spearman(
    "Excluding DeepSeek",
    without_deepseek,
)

without_both = sp[
    (sp.model != "deepseek/deepseek-v4-flash")
    & ~(
        (sp.model == "google/gemini-3.5-flash")
        & (sp.level == "L5")
    )
]

report_spearman(
    "Excluding DeepSeek and Gemini L5",
    without_both,
)


# ------------------------------------------------------------
# Part B. Deterministic query-disjoint folds
#
# A query is identified by (dataset, qid).
# Hashing makes the split deterministic and reproducible.
# ------------------------------------------------------------

analysis = ok[
    ok.model_id.isin(nonreasoning_models)
].copy()


def assign_fold(dataset, qid):
    key = f"{dataset}|{qid}".encode("utf-8")
    digest = hashlib.sha256(key).hexdigest()
    return int(digest[:8], 16) % 2


analysis["query_fold"] = [
    assign_fold(dataset, qid)
    for dataset, qid in zip(
        analysis.dataset,
        analysis.qid,
    )
]

query_folds = (
    analysis[
        ["dataset", "qid", "query_fold"]
    ]
    .drop_duplicates()
    .groupby("query_fold")
    .size()
)

print("\n" + "=" * 76)
print("QUERY FOLD COUNTS")
print("=" * 76)
print(query_folds.to_string())


# ------------------------------------------------------------
# Helper: mean κ across the three prompt paraphrases,
# matching the paper's validation calculation.
# ------------------------------------------------------------

contrast_levels = [1, 2, 4, 5]


def compute_level_kappas(frame):
    prompt_rows = []

    for (level, prompt_id), group in frame.groupby(
        ["politeness_level", "prompt_id"]
    ):
        kappa = cohen_kappa_score(
            group.human.astype(int),
            group.score.astype(int),
            weights="linear",
        )

        prompt_rows.append(
            {
                "level": int(level),
                "prompt_id": prompt_id,
                "kappa": kappa,
            }
        )

    prompt_frame = pd.DataFrame(prompt_rows)

    return (
        prompt_frame.groupby("level")
        .kappa.mean()
        .to_dict()
    )


# ------------------------------------------------------------
# Part C. Cross-fitting
#
# Direction 0->1:
#   Estimate Δ, D, and A using fold 0.
#   Compute κ changes using fold 1.
#
# Direction 1->0:
#   Estimate A using fold 1.
#   Compute κ changes using fold 0.
# ------------------------------------------------------------

crossfit_rows = []

for train_fold, test_fold in [(0, 1), (1, 0)]:
    direction = f"{train_fold}_to_{test_fold}"

    for model in nonreasoning_models:
        model_data = analysis[
            analysis.model_id == model
        ]

        train = model_data[
            model_data.query_fold == train_fold
        ]

        test = model_data[
            model_data.query_fold == test_fold
        ]

        # Predictor side: train-query score drift
        train_level_means = (
            train.groupby("politeness_level")
            .score.mean()
        )

        train_human_mean = (
            train[
                ["dataset", "qid", "docid", "human"]
            ]
            .drop_duplicates()
            .human.mean()
        )

        neutral_train_score = train_level_means.loc[3]

        delta_train = (
            neutral_train_score
            - train_human_mean
        )

        # Outcome side: held-out-query κ
        test_kappas = compute_level_kappas(test)
        neutral_test_kappa = test_kappas[3]

        for level in contrast_levels:
            drift_train = (
                train_level_means.loc[level]
                - neutral_train_score
            )

            alignment_change_train = (
                abs(delta_train + drift_train)
                - abs(delta_train)
            )

            dkappa_test = (
                test_kappas[level]
                - neutral_test_kappa
            )

            crossfit_rows.append(
                {
                    "model": model,
                    "direction": direction,
                    "train_fold": train_fold,
                    "test_fold": test_fold,
                    "level": f"L{level}",
                    "level_number": level,
                    "Delta_train": delta_train,
                    "D_train": drift_train,
                    "A_train": alignment_change_train,
                    "dkappa_test": dkappa_test,
                }
            )

crossfit = pd.DataFrame(crossfit_rows)

assert len(crossfit) == 56, (
    f"Expected 56 cross-fitted cells, found {len(crossfit)}"
)

print("\n" + "=" * 76)
print("QUERY-DISJOINT CROSS-FITTED CELLS")
print("=" * 76)

print(
    crossfit.to_string(
        index=False,
        formatters={
            "Delta_train": "{:+.6f}".format,
            "D_train": "{:+.6f}".format,
            "A_train": "{:+.6f}".format,
            "dkappa_test": "{:+.6f}".format,
        },
    )
)


# ------------------------------------------------------------
# Part D. Descriptive cross-fitted Spearman results
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("QUERY-DISJOINT SPEARMAN RESULTS")
print("=" * 76)

for direction in ["0_to_1", "1_to_0"]:
    subset = crossfit[
        crossfit.direction == direction
    ]

    result = spearmanr(
        subset.A_train,
        subset.dkappa_test,
    )

    print(
        f"{direction:12s} "
        f"n={len(subset):2d} "
        f"rho={result.statistic:+.6f} "
        f"nominal_p={result.pvalue:.8f}"
    )

combined_result = spearmanr(
    crossfit.A_train,
    crossfit.dkappa_test,
)

print(
    f"{'combined':12s} "
    f"n={len(crossfit):2d} "
    f"rho={combined_result.statistic:+.6f} "
    f"nominal_p={combined_result.pvalue:.8f}"
)


# ------------------------------------------------------------
# Part E. Exact permutation of complete model blocks
#
# For each model, its 8 cross-fitted observations
# (2 directions × 4 levels) remain together.
# ------------------------------------------------------------

def exact_model_block_test(frame):
    ordering = [
        ("0_to_1", 1),
        ("0_to_1", 2),
        ("0_to_1", 4),
        ("0_to_1", 5),
        ("1_to_0", 1),
        ("1_to_0", 2),
        ("1_to_0", 4),
        ("1_to_0", 5),
    ]

    A_blocks = {}
    Y_blocks = {}

    for model in nonreasoning_models:
        model_frame = frame[
            frame.model == model
        ].copy()

        model_frame["order_key"] = list(
            zip(
                model_frame.direction,
                model_frame.level_number,
            )
        )

        model_frame = (
            model_frame
            .set_index("order_key")
            .loc[ordering]
        )

        A_blocks[model] = (
            model_frame.A_train.to_numpy()
        )

        Y_blocks[model] = (
            model_frame.dkappa_test.to_numpy()
        )

    observed = spearmanr(
        np.concatenate(
            [A_blocks[m] for m in nonreasoning_models]
        ),
        np.concatenate(
            [Y_blocks[m] for m in nonreasoning_models]
        ),
    ).statistic

    permutation_results = []

    for permutation in itertools.permutations(
        nonreasoning_models
    ):
        permuted_A = np.concatenate(
            [A_blocks[m] for m in permutation]
        )

        fixed_Y = np.concatenate(
            [Y_blocks[m] for m in nonreasoning_models]
        )

        permutation_results.append(
            spearmanr(
                permuted_A,
                fixed_Y,
            ).statistic
        )

    permutation_results = np.asarray(
        permutation_results
    )

    one_sided_p = np.mean(
        permutation_results
        <= observed + 1e-15
    )

    two_sided_p = np.mean(
        np.abs(permutation_results)
        >= abs(observed) - 1e-15
    )

    return (
        observed,
        one_sided_p,
        two_sided_p,
        len(permutation_results),
    )


observed, block_p_one, block_p_two, n_permutations = (
    exact_model_block_test(crossfit)
)

print("\n" + "=" * 76)
print("QUERY-DISJOINT MODEL-BLOCK PERMUTATION")
print("=" * 76)

print("Number of exact permutations:", n_permutations)
print(f"Observed cross-fitted rho: {observed:+.6f}")
print(
    f"Exact one-sided block-permutation p: "
    f"{block_p_one:.8f}"
)
print(
    f"Exact two-sided block-permutation p: "
    f"{block_p_two:.8f}"
)


# ------------------------------------------------------------
# Part F. Model-level directional consistency
# ------------------------------------------------------------

crossfit_model_rhos = {}

print("\nWithin-model cross-fitted correlations:")

for model in nonreasoning_models:
    subset = crossfit[
        crossfit.model == model
    ]

    model_rho = spearmanr(
        subset.A_train,
        subset.dkappa_test,
    ).statistic

    crossfit_model_rhos[model] = model_rho

    print(
        f"{model:42s} "
        f"rho={model_rho:+.6f}"
    )

n_negative = sum(
    rho < 0
    for rho in crossfit_model_rhos.values()
)

crossfit_sign_test = binomtest(
    n_negative,
    len(crossfit_model_rhos),
    p=0.5,
    alternative="greater",
)

print(
    f"\nNegative model-level cross-fitted associations: "
    f"{n_negative}/{len(crossfit_model_rhos)}"
)

print(
    f"Exact one-sided sign-test p: "
    f"{crossfit_sign_test.pvalue:.8f}"
)

In [ ]:
# ============================================================
# AUDIT 3:
# Query-clustered bootstrap confidence intervals for DeepSeek
# Δκ at L1, L2, L4, and L5 relative to neutral L3.
#
# Assumes AUDIT 1/2 have been run and `analysis` is available.
# No model calls are made.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import cohen_kappa_score


DEEPSEEK = "deepseek/deepseek-v4-flash"
BOOTSTRAP_REPS = 2000
RANDOM_SEED = 20260814

ds = analysis[
    analysis.model_id == DEEPSEEK
].copy()

ds["query_key"] = (
    ds.dataset.astype(str)
    + "|"
    + ds.qid.astype(str)
)

query_keys = sorted(ds.query_key.unique())
query_to_index = {
    query_key: index
    for index, query_key in enumerate(query_keys)
}

ds["query_index"] = ds.query_key.map(
    query_to_index
).astype(int)

n_queries = len(query_keys)

print("=" * 72)
print("DEEPSEEK QUERY-CLUSTERED BOOTSTRAP")
print("=" * 72)
print("Rows:", len(ds))
print("Queries:", n_queries)
print("Prompts:", ds.prompt_id.nunique())
print("Runs:", ds.run.nunique())


# ------------------------------------------------------------
# Precompute the arrays for each level × prompt.
# Every bootstrap replicate only changes query weights.
# ------------------------------------------------------------

groups = {}

for (level, prompt_id), group in ds.groupby(
    ["politeness_level", "prompt_id"]
):
    groups[(int(level), prompt_id)] = {
        "human": group.human.astype(int).to_numpy(),
        "score": group.score.astype(int).to_numpy(),
        "query_index": group.query_index.to_numpy(),
    }


def calculate_level_kappas(query_counts):
    """
    query_counts[q] is the number of times query q was sampled.
    Returns mean κ across the three paraphrases at each level.
    """

    result = {}

    for level in [1, 2, 3, 4, 5]:
        prompt_kappas = []

        prompt_ids = sorted(
            prompt_id
            for current_level, prompt_id in groups
            if current_level == level
        )

        for prompt_id in prompt_ids:
            group = groups[(level, prompt_id)]

            sample_weights = query_counts[
                group["query_index"]
            ]

            kappa = cohen_kappa_score(
                group["human"],
                group["score"],
                weights="linear",
                sample_weight=sample_weights,
            )

            prompt_kappas.append(kappa)

        result[level] = np.mean(prompt_kappas)

    return result


# ------------------------------------------------------------
# Observed estimates
# ------------------------------------------------------------

observed_counts = np.ones(
    n_queries,
    dtype=int,
)

observed_kappa = calculate_level_kappas(
    observed_counts
)

observed_dkappa = {
    level: observed_kappa[level] - observed_kappa[3]
    for level in [1, 2, 4, 5]
}

print("\nObserved mean κ across paraphrases:")
for level in [1, 2, 3, 4, 5]:
    print(
        f"L{level}: {observed_kappa[level]:.6f}"
    )

print("\nObserved Δκ relative to L3:")
for level in [1, 2, 4, 5]:
    print(
        f"L{level}: {observed_dkappa[level]:+.6f}"
    )


# ------------------------------------------------------------
# Paired query-clustered bootstrap
# ------------------------------------------------------------

rng = np.random.default_rng(RANDOM_SEED)

bootstrap_dkappa = {
    1: [],
    2: [],
    4: [],
    5: [],
}

for bootstrap_index in range(BOOTSTRAP_REPS):
    sampled_queries = rng.integers(
        low=0,
        high=n_queries,
        size=n_queries,
    )

    query_counts = np.bincount(
        sampled_queries,
        minlength=n_queries,
    )

    bootstrap_kappa = calculate_level_kappas(
        query_counts
    )

    for level in [1, 2, 4, 5]:
        bootstrap_dkappa[level].append(
            bootstrap_kappa[level]
            - bootstrap_kappa[3]
        )

    if (bootstrap_index + 1) % 500 == 0:
        print(
            f"Completed "
            f"{bootstrap_index + 1}/"
            f"{BOOTSTRAP_REPS} replicates"
        )


# ------------------------------------------------------------
# Percentile confidence intervals
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("QUERY-CLUSTERED 95% BOOTSTRAP INTERVALS")
print("=" * 72)

summary_rows = []

for level in [1, 2, 4, 5]:
    values = np.asarray(
        bootstrap_dkappa[level]
    )

    lower, median, upper = np.quantile(
        values,
        [0.025, 0.5, 0.975],
    )

    probability_nonpositive = np.mean(
        values <= 0
    )

    summary_rows.append(
        {
            "contrast": f"L{level} - L3",
            "observed_dkappa": observed_dkappa[level],
            "bootstrap_median": median,
            "ci_2.5%": lower,
            "ci_97.5%": upper,
            "bootstrap_P(dkappa<=0)": probability_nonpositive,
        }
    )

summary = pd.DataFrame(summary_rows)

print(
    summary.to_string(
        index=False,
        formatters={
            "observed_dkappa": "{:+.6f}".format,
            "bootstrap_median": "{:+.6f}".format,
            "ci_2.5%": "{:+.6f}".format,
            "ci_97.5%": "{:+.6f}".format,
            "bootstrap_P(dkappa<=0)": "{:.6f}".format,
        },
    )
)

In [ ]:
# ============================================================
# AUDIT 4:
# Query-level ranking analysis with paired bootstrap intervals
#
# Ranking construction:
#   1. Mean score across prompts and runs
#   2. Score descending
#   3. BM25 descending as deterministic tie-break
#   4. docid ascending as final deterministic tie-break
#
# Assumes AUDIT 1 has run and `ok`, `qrels`,
# `nonreasoning_models`, and `reasoning_model` are available.
# ============================================================

import numpy as np
import pandas as pd

from scipy.stats import kendalltau


BOOTSTRAP_REPS = 2000
RANDOM_SEED = 20260814

all_models = (
    nonreasoning_models
    + [reasoning_model]
)

ranking_data = ok[
    ok.model_id.isin(all_models)
].copy()


# ------------------------------------------------------------
# 1. Load BM25 scores for deterministic tie-breaking
# ------------------------------------------------------------

bm25 = {}

for dataset in ["dl19", "dl20"]:
    path = f"{BASE}/inputs/bm25_{dataset}.txt"

    with open(path) as file:
        for line in file:
            qid, _, docid, rank, score, run_name = (
                line.split()
            )

            bm25[
                (
                    dataset,
                    str(qid),
                    str(docid),
                )
            ] = float(score)

print("=" * 76)
print("RANKING UNCERTAINTY ANALYSIS")
print("=" * 76)
print("BM25 entries:", len(bm25))


# ------------------------------------------------------------
# 2. Ranking and metric helpers
# ------------------------------------------------------------

def make_ranking(frame, dataset, qid):
    """
    Average across all prompt paraphrases and runs at one
    politeness level, then rank with deterministic tie-breaking.
    """

    scores = (
        frame.groupby("docid")
        .score.mean()
        .reset_index()
    )

    scores["bm25"] = [
        bm25.get(
            (dataset, str(qid), str(docid)),
            0.0,
        )
        for docid in scores.docid
    ]

    scores = scores.sort_values(
        ["score", "bm25", "docid"],
        ascending=[False, False, True],
        kind="mergesort",
    )

    return scores.docid.astype(str).tolist()


def dcg(relevances):
    return sum(
        (2 ** relevance - 1)
        / np.log2(rank + 2)
        for rank, relevance in enumerate(relevances)
    )


def ndcg_at_10(ranked_docids, dataset, qid):
    retrieved_relevances = [
        qrels.get(
            (dataset, str(qid), str(docid)),
            0,
        )
        for docid in ranked_docids[:10]
    ]

    all_candidate_relevances = [
        qrels.get(
            (dataset, str(qid), str(docid)),
            0,
        )
        for docid in ranked_docids
    ]

    ideal_relevances = sorted(
        all_candidate_relevances,
        reverse=True,
    )[:10]

    ideal_dcg = dcg(ideal_relevances)

    if ideal_dcg == 0:
        return np.nan

    return (
        dcg(retrieved_relevances)
        / ideal_dcg
    )


def ranking_kendall_tau(
    tone_ranking,
    neutral_ranking,
):
    """
    Both rankings are total orders after deterministic tie-breaking.
    """

    neutral_positions = {
        docid: position
        for position, docid
        in enumerate(neutral_ranking)
    }

    common_docids = [
        docid
        for docid in tone_ranking
        if docid in neutral_positions
    ]

    if len(common_docids) < 3:
        return np.nan

    tone_positions = np.arange(
        len(common_docids)
    )

    corresponding_neutral_positions = [
        neutral_positions[docid]
        for docid in common_docids
    ]

    return kendalltau(
        tone_positions,
        corresponding_neutral_positions,
    ).statistic


# ------------------------------------------------------------
# 3. Calculate query-level metrics
# ------------------------------------------------------------

query_metric_rows = []

for model in all_models:
    model_data = ranking_data[
        ranking_data.model_id == model
    ]

    for (dataset, qid), query_frame in model_data.groupby(
        ["dataset", "qid"]
    ):
        rankings = {}
        ndcgs = {}

        for level in [1, 2, 3, 4, 5]:
            level_frame = query_frame[
                query_frame.politeness_level == level
            ]

            if level_frame.empty:
                continue

            ranking = make_ranking(
                level_frame,
                dataset,
                qid,
            )

            rankings[level] = ranking
            ndcgs[level] = ndcg_at_10(
                ranking,
                dataset,
                qid,
            )

        if 3 not in rankings:
            continue

        for level in [1, 2, 4, 5]:
            if level not in rankings:
                continue

            tau = ranking_kendall_tau(
                rankings[level],
                rankings[3],
            )

            dndcg = (
                ndcgs[level]
                - ndcgs[3]
            )

            query_metric_rows.append(
                {
                    "model": model,
                    "dataset": dataset,
                    "qid": str(qid),
                    "level": f"L{level}",
                    "level_number": level,
                    "dndcg10": dndcg,
                    "tau": tau,
                }
            )

query_metrics = pd.DataFrame(
    query_metric_rows
)

print(
    "Query-level model/contrast rows:",
    len(query_metrics),
)

print(
    "Queries per model/contrast:"
)

print(
    query_metrics.groupby(
        ["model", "level"]
    )
    .size()
    .groupby("model")
    .agg(["min", "max"])
    .to_string()
)


# ------------------------------------------------------------
# 4. Paired query bootstrap for each model × contrast
# ------------------------------------------------------------

rng = np.random.default_rng(
    RANDOM_SEED
)

summary_rows = []

for (model, level), frame in query_metrics.groupby(
    ["model", "level"]
):
    frame = frame[
        frame.dndcg10.notna()
        & frame.tau.notna()
    ].copy()

    dndcg_values = frame.dndcg10.to_numpy()
    tau_values = frame.tau.to_numpy()

    n_queries = len(frame)

    bootstrap_indices = rng.integers(
        low=0,
        high=n_queries,
        size=(BOOTSTRAP_REPS, n_queries),
    )

    bootstrap_dndcg = (
        dndcg_values[bootstrap_indices]
        .mean(axis=1)
    )

    bootstrap_tau = (
        tau_values[bootstrap_indices]
        .mean(axis=1)
    )

    dndcg_lower, dndcg_median, dndcg_upper = (
        np.quantile(
            bootstrap_dndcg,
            [0.025, 0.5, 0.975],
        )
    )

    tau_lower, tau_median, tau_upper = (
        np.quantile(
            bootstrap_tau,
            [0.025, 0.5, 0.975],
        )
    )

    summary_rows.append(
        {
            "model": model,
            "level": level,
            "n_queries": n_queries,
            "mean_dndcg10": dndcg_values.mean(),
            "dndcg_ci_low": dndcg_lower,
            "dndcg_ci_high": dndcg_upper,
            "mean_tau": tau_values.mean(),
            "tau_ci_low": tau_lower,
            "tau_ci_high": tau_upper,
        }
    )

ranking_summary = pd.DataFrame(
    summary_rows
)

ranking_summary["abs_dndcg10"] = (
    ranking_summary.mean_dndcg10.abs()
)

ranking_summary = ranking_summary.sort_values(
    ["model", "level"]
)


# ------------------------------------------------------------
# 5. Full output
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("QUERY-CLUSTERED RANKING INTERVALS")
print("=" * 76)

print(
    ranking_summary[
        [
            "model",
            "level",
            "n_queries",
            "mean_dndcg10",
            "dndcg_ci_low",
            "dndcg_ci_high",
            "mean_tau",
            "tau_ci_low",
            "tau_ci_high",
        ]
    ].to_string(
        index=False,
        formatters={
            "mean_dndcg10": "{:+.6f}".format,
            "dndcg_ci_low": "{:+.6f}".format,
            "dndcg_ci_high": "{:+.6f}".format,
            "mean_tau": "{:.6f}".format,
            "tau_ci_low": "{:.6f}".format,
            "tau_ci_high": "{:.6f}".format,
        },
    )
)


# ------------------------------------------------------------
# 6. Identify the largest NDCG change and lowest tau
# ------------------------------------------------------------

largest_dndcg = ranking_summary.loc[
    ranking_summary.abs_dndcg10.idxmax()
]

lowest_tau = ranking_summary.loc[
    ranking_summary.mean_tau.idxmin()
]

print("\n" + "=" * 76)
print("EXTREME RANKING RESULTS")
print("=" * 76)

print("\nLargest absolute mean ΔNDCG@10:")
print(
    largest_dndcg[
        [
            "model",
            "level",
            "n_queries",
            "mean_dndcg10",
            "dndcg_ci_low",
            "dndcg_ci_high",
        ]
    ].to_string()
)

print("\nLowest mean Kendall tau:")
print(
    lowest_tau[
        [
            "model",
            "level",
            "n_queries",
            "mean_tau",
            "tau_ci_low",
            "tau_ci_high",
        ]
    ].to_string()
)

ranking_summary["dndcg_ci_excludes_zero"] = (
    (ranking_summary.dndcg_ci_low > 0)
    | (ranking_summary.dndcg_ci_high < 0)
)

print(
    "\nContrasts whose ΔNDCG@10 CI excludes zero:",
    int(
        ranking_summary
        .dndcg_ci_excludes_zero
        .sum()
    ),
    "/",
    len(ranking_summary),
)

In [ ]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# 避免 Matplotlib 在 PDF 中生成 Type-3 字体
plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# ------------------------------------------------------------
# Final recomputed data from the final parquet
#
# Each model:
#   Delta = mean score at L3 - mean human qrel
#   level -> (D(level), Delta-kappa)
# ------------------------------------------------------------

data = {
    "DeepSeek V4 Flash": (
        0.543120,
        {
            "L1": (-0.148403, +0.054067),
            "L2": (-0.078902, +0.027762),
            "L4": (-0.094054, +0.034139),
            "L5": (-0.134410, +0.053490),
        },
    ),
    "GPT-5.4-mini": (
        0.708548,
        {
            "L1": (-0.045788, +0.006434),
            "L2": (+0.004908, -0.014663),
            "L4": (+0.000048, -0.002848),
            "L5": (-0.015294, -0.002575),
        },
    ),
    "Claude Haiku 4.5": (
        0.519535,
        {
            "L1": (-0.010816, -0.000622),
            "L2": (+0.013246, -0.009867),
            "L4": (-0.021107, -0.006531),
            "L5": (-0.010244, -0.010016),
        },
    ),
    "Qwen3.7-Plus": (
        0.395321,
        {
            "L1": (-0.074662, -0.007661),
            "L2": (+0.012055, -0.014808),
            "L4": (+0.037554, -0.032113),
            "L5": (-0.008433, -0.003880),
        },
    ),
    "Gemini 3.5 Flash": (
        0.479750,
        {
            "L1": (-0.051934, +0.013418),
            "L2": (+0.000524, -0.007189),
            "L4": (-0.015104, +0.011786),
            "L5": (+0.038212, -0.047391),
        },
    ),
    "GPT-5.5": (
        0.661898,
        {
            "L1": (-0.009061, +0.001138),
            "L2": (+0.016929, -0.003669),
            "L4": (+0.030520, -0.005958),
            "L5": (+0.030043, -0.008128),
        },
    ),
    "Claude Opus 4.8": (
        0.352170,
        {
            "L1": (-0.038150, +0.010961),
            "L2": (+0.005961, -0.015134),
            "L4": (-0.001669, -0.008743),
            "L5": (+0.025274, -0.011920),
        },
    ),
}

level_order = ["L1", "L2", "L4", "L5"]

# ------------------------------------------------------------
# Construct the 28 model-tone cells
# A(level) = |Delta + D(level)| - |Delta|
# ------------------------------------------------------------

rows = []

for model, (delta, levels) in data.items():
    for level in level_order:
        drift, dkappa = levels[level]
        alignment_change = abs(delta + drift) - abs(delta)

        rows.append({
            "model": model,
            "level": level,
            "Delta": delta,
            "D": drift,
            "A": alignment_change,
            "dkappa": dkappa,
        })

sp = pd.DataFrame(rows)

# Full-data descriptive Spearman correlation
rho = spearmanr(sp["A"], sp["dkappa"]).statistic

# Sanity check against the final recomputed result
assert np.isclose(rho, -0.7985769020, atol=1e-9)

# ------------------------------------------------------------
# Optional diagnostic: exact whole-model block permutation
#
# This preserves the four tone contrasts within each model.
# Do not put this p-value in the plot title. The paper's primary
# inferential result is the query-disjoint cross-fit p = 0.0187.
# ------------------------------------------------------------

models = list(data.keys())

a_blocks = {
    model: (
        sp[sp["model"] == model]
        .set_index("level")
        .loc[level_order, "A"]
        .to_numpy()
    )
    for model in models
}

fixed_dkappa = np.concatenate([
    sp[sp["model"] == model]
    .set_index("level")
    .loc[level_order, "dkappa"]
    .to_numpy()
    for model in models
])

permuted_rhos = []

for permutation in itertools.permutations(models):
    permuted_A = np.concatenate([
        a_blocks[model] for model in permutation
    ])

    permuted_rhos.append(
        spearmanr(permuted_A, fixed_dkappa).statistic
    )

permuted_rhos = np.asarray(permuted_rhos)

# Pre-specified direction: rho < 0
full_data_block_p = np.mean(permuted_rhos <= rho)

# ------------------------------------------------------------
# Sensitivity estimates used in the caption/text
# ------------------------------------------------------------

without_gemini_l5 = ~(
    (sp["model"] == "Gemini 3.5 Flash") &
    (sp["level"] == "L5")
)

without_deepseek = sp["model"] != "DeepSeek V4 Flash"

without_both = without_gemini_l5 & without_deepseek

rho_without_gemini_l5 = spearmanr(
    sp.loc[without_gemini_l5, "A"],
    sp.loc[without_gemini_l5, "dkappa"],
).statistic

rho_without_deepseek = spearmanr(
    sp.loc[without_deepseek, "A"],
    sp.loc[without_deepseek, "dkappa"],
).statistic

rho_without_both = spearmanr(
    sp.loc[without_both, "A"],
    sp.loc[without_both, "dkappa"],
).statistic

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

markers = {
    "GPT-5.4-mini": "o",
    "Claude Haiku 4.5": "s",
    "Qwen3.7-Plus": "^",
    "Gemini 3.5 Flash": "D",
    "GPT-5.5": "v",
    "Claude Opus 4.8": "P",
}

from matplotlib.ticker import MaxNLocator, FormatStrFormatter

# ACM single-column native size
fig, ax = plt.subplots(figsize=(3.35, 3.10))

for model in models:
    model_rows = sp[sp["model"] == model]

    if model == "DeepSeek V4 Flash":
        ax.scatter(
            model_rows["A"],
            model_rows["dkappa"],
            s=78,
            c="crimson",
            marker="*",
            edgecolors="black",
            linewidths=0.45,
            label=model,
            zorder=3,
        )
    else:
        ax.scatter(
            model_rows["A"],
            model_rows["dkappa"],
            s=30,
            c="steelblue",
            marker=markers[model],
            alpha=0.82,
            edgecolors="none",
            label=model,
            zorder=2,
        )

# Illustrative linear trend
slope, intercept = np.polyfit(sp["A"], sp["dkappa"], 1)
xs = np.linspace(sp["A"].min(), sp["A"].max(), 100)

ax.plot(
    xs,
    slope * xs + intercept,
    "--",
    color="gray",
    linewidth=1.0,
    label="trend",
    zorder=1,
)

ax.axhline(0, color="lightgray", linewidth=0.6, zorder=0)
ax.axvline(0, color="lightgray", linewidth=0.6, zorder=0)

ax.set_xlabel(
    r"Alignment change $A(\ell)$",
    fontsize=8,
)

# Shorter label saves substantial horizontal space
ax.set_ylabel(
    r"Agreement change $\Delta\kappa$",
    fontsize=8,
)

ax.tick_params(labelsize=7)

# Fewer digits/ticks remain readable at single-column width
ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
ax.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))

# Remove title: rho is already reported in the figure caption
ax.legend(
    fontsize=6.2,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.22),
    ncol=2,
    frameon=False,
    handletextpad=0.4,
    columnspacing=0.8,
    labelspacing=0.35,
)

fig.subplots_adjust(
    left=0.20,
    right=0.98,
    top=0.97,
    bottom=0.34,
)

fig.savefig(
    "/content/drive/MyDrive/Projects/research/"
    "politeness-llm/paper/fig_spearman_rerun2.pdf"
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, FormatStrFormatter

# Embed TrueType fonts instead of Type 3 fonts
plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# Final recomputed Delta-kappa values relative to L3
dk_data = {
    "DeepSeek V4 Flash": {
        "L1": +0.054067,
        "L2": +0.027762,
        "L3":  0.000000,
        "L4": +0.034139,
        "L5": +0.053490,
    },
    "GPT-5.4-mini": {
        "L1": +0.006434,
        "L2": -0.014663,
        "L3":  0.000000,
        "L4": -0.002848,
        "L5": -0.002575,
    },
    "Claude Haiku 4.5": {
        "L1": -0.000622,
        "L2": -0.009867,
        "L3":  0.000000,
        "L4": -0.006531,
        "L5": -0.010016,
    },
    "Qwen3.7-Plus": {
        "L1": -0.007661,
        "L2": -0.014808,
        "L3":  0.000000,
        "L4": -0.032113,
        "L5": -0.003880,
    },
    "Gemini 3.5 Flash": {
        "L1": +0.013418,
        "L2": -0.007189,
        "L3":  0.000000,
        "L4": +0.011786,
        "L5": -0.047391,
    },
    "GPT-5.5": {
        "L1": +0.001138,
        "L2": -0.003669,
        "L3":  0.000000,
        "L4": -0.005958,
        "L5": -0.008128,
    },
    "Claude Opus 4.8": {
        "L1": +0.010961,
        "L2": -0.015134,
        "L3":  0.000000,
        "L4": -0.008743,
        "L5": -0.011920,
    },
}

levels = ["L1", "L2", "L3", "L4", "L5"]
x = np.arange(len(levels))

# Distinct markers make the figure readable without color
markers = {
    "DeepSeek V4 Flash": "*",
    "GPT-5.4-mini": "o",
    "Claude Haiku 4.5": "s",
    "Qwen3.7-Plus": "^",
    "Gemini 3.5 Flash": "D",
    "GPT-5.5": "v",
    "Claude Opus 4.8": "P",
}

# Shorter display names fit a single-column legend
display_names = {
    "DeepSeek V4 Flash": "DeepSeek",
    "GPT-5.4-mini": "GPT-5.4-mini",
    "Claude Haiku 4.5": "Haiku 4.5",
    "Qwen3.7-Plus": "Qwen3.7-Plus",
    "Gemini 3.5 Flash": "Gemini Flash",
    "GPT-5.5": "GPT-5.5",
    "Claude Opus 4.8": "Opus 4.8",
}

# Native ACM single-column size
fig, ax = plt.subplots(figsize=(3.35, 2.95))

for model, values in dk_data.items():
    ys = [values[level] for level in levels]

    if model == "DeepSeek V4 Flash":
        ax.plot(
            x,
            ys,
            color="crimson",
            linewidth=1.8,
            marker=markers[model],
            markersize=7.0,
            markeredgecolor="black",
            markeredgewidth=0.35,
            label=display_names[model],
            zorder=3,
        )
    else:
        ax.plot(
            x,
            ys,
            color="steelblue",
            linewidth=0.9,
            alpha=0.72,
            marker=markers[model],
            markersize=3.8,
            label=display_names[model],
            zorder=2,
        )

ax.axhline(
    0,
    color="gray",
    linewidth=0.7,
    linestyle="--",
    zorder=1,
)

ax.set_xticks(x)
ax.set_xticklabels(levels, fontsize=7)

ax.set_xlabel(
    r"Tone level (L1 rude $\rightarrow$ L5 deferential)",
    fontsize=8,
)

ax.set_ylabel(
    r"Agreement change $\Delta\kappa$",
    fontsize=8,
)

# Remove the internal title; the caption already explains the result
ax.tick_params(axis="y", labelsize=7)
ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))

ax.legend(
    fontsize=6.1,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.22),
    ncol=2,
    frameon=False,
    handletextpad=0.4,
    columnspacing=0.8,
    labelspacing=0.35,
)

ax.margins(x=0.03)

fig.subplots_adjust(
    left=0.20,
    right=0.98,
    top=0.98,
    bottom=0.34,
)

output_path = (
    "/content/drive/MyDrive/Projects/research/"
    "politeness-llm/paper/fig_dkappa_rerun1.pdf"
)

fig.savefig(output_path)

print(f"Saved: {output_path}")
plt.show()

## 9. 决策错误率（DE）

顺序：h9a selftest + fit-lodo-rules + paste（盲态）→ 把 `/content/de_paste.txt` 的块填进 `paper/PREDICTIONS.md`（`# DE 预注册 — 决策错误率`）并 commit → h9b 开箱（门控：标题在干净已提交的 PREDICTIONS.md 中、输入 sha256 与登记一致；dl22 需 `--unseal-dl22`）→ h9c（可选）local-calib。全部 CPU，无 API 调用。
d1-b5 的完整性核对需要内部仓库的 `paper/sigir27/tables/b5_per_system_ndcg.csv`：先复制到 `$DATA_DIR/derived/decision_errors/`（paste 前，以便登记其 sha256）。

In [ ]:
# (h9a) DE step 1 (blind): selftest, LODO msmarco rule (dl19+dl20, dial in {L1,L3}), registration block -> fill + COMMIT PREDICTIONS.md before h9b
%%bash
cd /content/politeness-llm && git pull -q && git log --oneline -1
python scripts/decision_errors.py selftest > /content/de_selftest.txt 2>&1; echo SELFTEST_EXIT=$?; tail -2 /content/de_selftest.txt
python scripts/decision_errors.py fit-lodo-rules > /content/de_lodo.txt 2>&1; echo LODO_EXIT=$?; tail -8 /content/de_lodo.txt
python scripts/decision_errors.py paste > /content/de_paste.txt 2>&1; echo PASTE_EXIT=$?; wc -l /content/de_paste.txt
cp /content/de_lodo.txt /content/de_paste.txt $DATA_DIR/derived/decision_errors/

In [ ]:
# (h9b) DE UNBOXING — ONLY after the DE registration commit (gate checks heading, clean commit, sha256s; dl22 needs --unseal-dl22)
%%bash
cd /content/politeness-llm && git pull -q && git log --oneline -1
D="python scripts/decision_errors.py"
$D d2-pool --datasets dl22,dl21,antique --unseal-dl22 > /content/de_d2.txt 2>&1; echo D2_EXIT=$?; sed -n '/BACKFILL/,$p' /content/de_d2.txt | head -16
$D d1-b5 > /content/de_d1b5.txt 2>&1; echo D1B5_EXIT=$?; sed -n '/BACKFILL/,$p' /content/de_d1b5.txt | head -10
$D d3-validate --unseal-dl22 > /content/de_d3.txt 2>&1; echo D3_EXIT=$?; sed -n '/BACKFILL/,$p' /content/de_d3.txt | head -8
$D d1-students --unseal-dl22 > /content/de_d1students.txt 2>&1; echo D1S_EXIT=$?; sed -n '/BACKFILL/,$p' /content/de_d1students.txt | head -6
cp /content/de_d2.txt /content/de_d1b5.txt /content/de_d3.txt /content/de_d1students.txt $DATA_DIR/derived/decision_errors/

In [ ]:
# (h9c) OPTIONAL local-calib (Priority 5): SHIFT_n / METRIC_SHIFT_n, n=200 calibration pairs, R=200 query-disjoint splits, every arm on held-out queries only
%%bash
cd /content/politeness-llm && git pull -q && git log --oneline -1
python scripts/decision_errors.py local-calib --studies d2,d1-b5 --datasets dl22,dl21 --unseal-dl22 > /content/de_lc.txt 2>&1; echo LC_EXIT=$?; tail -3 /content/de_lc.txt
cp /content/de_lc.txt $DATA_DIR/derived/decision_errors/

In [ ]:
# (h9d) DE add-ons (pre-announced "DE 探索性追加 2"; exploratory, descriptive): RETHRESH + trivial baselines (D2/D3), D3 cluster bootstrap + label classes, D1a constant-offset curve, JSON meta
%%bash
cd /content/politeness-llm && git pull -q && git log --oneline -1
python scripts/decision_errors.py addons --unseal-dl22 > /content/de_addons.txt 2>&1; echo ADDONS_EXIT=$?
cp /content/de_addons.txt $DATA_DIR/derived/decision_errors/
tail -25 /content/de_addons.txt

## 10. 标注不完整性检查（IC）

顺序：h10a selftest + paste（盲态：只有计数与 sha256）→ 把 `/content/ic_paste.txt` 的块填进 `paper/PREDICTIONS.md`（`# IC 预注册 — 标注不完整性检查`）并 commit → h10b 开箱（门控：标题在干净已提交的 PREDICTIONS.md 中、输入 sha256 与登记一致；dl22 需 `--unseal-dl22`）。全部 CPU，无 API 调用；输出在 `$DATA_DIR/derived/incompleteness/`。

In [ ]:
# (h10a) IC step 1 (blind): selftest + registration block (counts and sha256 only) -> fill + COMMIT PREDICTIONS.md before h10b
%%bash
cd /content/politeness-llm && git pull -q && git log --oneline -1
mkdir -p $DATA_DIR/derived/incompleteness
python scripts/incompleteness_check.py selftest > /content/ic_selftest.txt 2>&1; echo SELFTEST_EXIT=$?; tail -2 /content/ic_selftest.txt
python scripts/incompleteness_check.py paste > /content/ic_paste.txt 2>&1; echo PASTE_EXIT=$?; wc -l /content/ic_paste.txt
cp /content/ic_selftest.txt /content/ic_paste.txt $DATA_DIR/derived/incompleteness/

In [ ]:
# (h10b) IC UNBOXING — ONLY after the IC registration commit (gate: heading, clean commit, sha256s; dl22 needs --unseal-dl22)
%%bash
cd /content/politeness-llm && git pull -q && git log --oneline -1
python scripts/incompleteness_check.py run --unseal-dl22 > /content/ic_run.txt 2>&1; echo RUN_EXIT=$?
cp /content/ic_run.txt $DATA_DIR/derived/incompleteness/
tail -40 /content/ic_run.txt